# LoRa Prediction and Optimization System

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import randint, uniform
from torch.utils.data import Dataset, DataLoader
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from pathlib import Path
import warnings
import ee
import time
import json
import os
from dotenv import load_dotenv
from typing import Dict, List, Tuple, Optional, Union
import pickle
from scipy.spatial.distance import cdist
import logging
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import heapq


### Hyperparameter Tuning with Optuna

In [4]:
# Optuna for hyperparameter tuning
try:
    import optuna
    from optuna.pruners import MedianPruner
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False
    logging.warning("Optuna not installed. Hyperparameter tuning will be disabled.")


### Environment Variables, Logging and Device Configuration

In [5]:
# Load environment variables
load_dotenv()
warnings.filterwarnings('ignore')

# Logging Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('lora_system.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


2025-10-19 05:47:06,955 - __main__ - INFO - Using device: cuda
2025-10-19 05:47:06,959 - __main__ - INFO - GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
2025-10-19 05:47:06,959 - __main__ - INFO - Memory Available: 6.44 GB


### Constants and Physical Parameters

In [6]:
# SNR thresholds for different spreading factors (from LoRaWAN specification)
SNR_THRESHOLD = {
    7: -7.5, 8: -10, 9: -12.5, 10: -15, 11: -17.5, 12: -20
}

# FIXED: Better calibrated decay constants
# Lower K = slower PDR recovery (more realistic for poor conditions)
LAND_COVER_TO_K = {
    10: 0.15,  # Tree cover - SLOW recovery
    20: 0.18,  # Shrubland
    30: 0.28,  # Grassland - GOOD
    40: 0.25,  # Cropland - GOOD
    50: 0.08,  # Built-up - VERY SLOW (realistic for buildings)
    60: 0.35,  # Bare/sparse - EXCELLENT
    70: 0.22,  # Snow and ice
    80: 0.40,  # Water - BEST
    90: 0.20,  # Herbaceous wetland
    95: 0.12,  # Mangroves - SLOW
    100: 0.25  # Moss and lichen
}

# FIXED: More realistic terrain penalties
PENALTY_MAP = {
    10: 0.7,   # Tree cover - HIGH penalty
    20: 0.5,   # Shrubland - MODERATE-HIGH
    30: 0.1,   # Grassland - LOW (best for open area)
    40: 0.15,  # Cropland - LOW
    50: 0.95,  # Built-up - VERY HIGH (realistic)
    60: 0.05,  # Bare/sparse - VERY LOW
    70: 0.6,   # Snow/ice - HIGH
    80: 0.0,   # Water - NO penalty (best)
    90: 0.4,   # Wetland - MODERATE
    95: 0.65,  # Mangroves - HIGH
    100: 0.2   # Moss/lichen - LOW
}


### Data Classes

In [7]:
@dataclass
class LoRaParameters:
    """LoRa communication parameters with validation"""
    tx_power: float = 14.0  # dBm (2-20)
    spreading_factor: int = 7  # (7-12)
    frequency: float = 868.0  # MHz
    bandwidth: float = 125.0  # kHz
    coding_rate: int = 4  # 4/5
    
    def __post_init__(self):
        """Validate parameters after initialization"""
        if not (2 <= self.tx_power <= 30):
            raise ValueError(f"TX power {self.tx_power} must be in range [2, 30] dBm")
        if self.spreading_factor not in [7, 8, 9, 10, 11, 12]:
            raise ValueError(f"Spreading factor {self.spreading_factor} must be in [7-12]")
        if not (100 <= self.frequency <= 1000):
            raise ValueError(f"Frequency {self.frequency} must be in range [100, 1000] MHz")

@dataclass
class PathPoint:
    """Represents a point in the path with all attributes"""
    lat: float
    lon: float
    elevation: float = 0.0
    land_cover: int = 50
    terrain_penalty: float = 0.5
    rssi: float = -120.0  # FIXED: More realistic default
    snr: float = -10.0    # FIXED: More realistic default
    pdr: float = 0.0      # FIXED: Start at 0, not 0.5
    path_loss: float = 120.0
    distance_to_start: float = 0.0
    distance_to_goal: float = 0.0
    grid_x: int = 0
    grid_y: int = 0
    is_relay: bool = False
    hop_number: int = 0
    # Path spatial features
    path_built_up_fraction: float = 0.0
    path_vegetation_fraction: float = 0.0
    path_water_fraction: float = 0.0
    path_avg_penalty: float = 0.5
    path_elevation_std: float = 0.0
    max_terrain_obstruction_m: float = 0.0
    path_dominant_land_cover: int = 50

@dataclass
class OptimizationConfig:
    """Configuration for path optimization"""
    grid_spacing_km: float = 1.5
    corridor_width_km: float = 4.0
    adaptive_grid: bool = True
    max_path_deviation: float = 0.5
    min_pdr_threshold: float = 0.3
    prefer_water: bool = True
    avoid_buildings: bool = True

@dataclass
class GEEConfig:
    """Configuration for Google Earth Engine integration"""
    batch_size: int = 50
    workers: int = 5
    retry_attempts: int = 3
    fallback_to_individual: bool = True
    cache_enabled: bool = True
    cache_file: str = 'gee_cache.pkl'
    path_spatial_samples: int = 15

@dataclass
class HyperparameterConfig:
    """Configuration for hyperparameter tuning"""
    enable: bool = False
    nn_trials: int = 40
    rf_n_iter: int = 40
    xgb_n_iter: int = 40
    cv_folds: int = 3
    tuning_data_ratio: float = 0.2


### Exceptions and Input Validations

In [8]:
class GEEDataUnavailableError(Exception):
    """Raised when Google Earth Engine data cannot be fetched"""
    pass

class InvalidCoordinatesError(Exception):
    """Raised when coordinates are out of valid range"""
    pass

class InvalidLoRaParametersError(Exception):
    """Raised when LoRa parameters are invalid"""
    pass

class NoViablePathError(Exception):
    """Raised when A* cannot find a path between start and destination"""
    pass

# Input Validation Functions
def validate_coordinates(lat: float, lon: float, name: str = "Point"):
    """Validate geographic coordinates"""
    if not isinstance(lat, (int, float)):
        raise InvalidCoordinatesError(f"{name} latitude must be a number, got {type(lat).__name__}")
    if not isinstance(lon, (int, float)):
        raise InvalidCoordinatesError(f"{name} longitude must be a number, got {type(lon).__name__}")
    
    if not (-90 <= lat <= 90):
        raise InvalidCoordinatesError(
            f"{name} latitude {lat} out of range [-90, 90]. "
            f"Did you swap latitude and longitude?"
        )
    if not (-180 <= lon <= 180):
        raise InvalidCoordinatesError(
            f"{name} longitude {lon} out of range [-180, 180]. "
            f"Did you swap latitude and longitude?"
        )

def validate_distance(start_lat: float, start_lon: float, dest_lat: float, dest_lon: float):
    """Validate that start and destination are not identical and not too far"""
    if start_lat == dest_lat and start_lon == dest_lon:
        raise InvalidCoordinatesError("Start and destination coordinates are identical")
    
    # Calculate distance
    R = 6371000  # Earth radius in meters
    phi1, phi2 = np.radians(start_lat), np.radians(dest_lat)
    dphi = np.radians(dest_lat - start_lat)
    dlambda = np.radians(dest_lon - start_lon)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    distance = R * c
    
    if distance < 100:  # Less than 100 meters
        raise InvalidCoordinatesError(
            f"Distance too short: {distance:.1f}m (minimum 100m). "
            f"Start and destination are almost identical."
        )
    if distance > (200 * 1000):  # More than 200 km
        logger.warning(
            f"Distance very large: {distance/1000:.1f}km - optimization may be slow. "
            f"Consider breaking into multiple segments."
        )

def validate_lora_parameters(spreading_factor: int, tx_power: int, frequency: int):
    """Validate LoRa communication parameters"""
    # Spreading Factor
    if not isinstance(spreading_factor, int):
        raise InvalidLoRaParametersError(
            f"spreading_factor must be an integer, got {type(spreading_factor).__name__}"
        )
    if spreading_factor not in [7, 8, 9, 10, 11, 12]:
        raise InvalidLoRaParametersError(
            f"spreading_factor {spreading_factor} invalid. Must be 7, 8, 9, 10, 11, or 12. "
            f"(SF7=shortest range/fastest, SF12=longest range/slowest)"
        )
    
    # TX Power
    if not isinstance(tx_power, int):
        raise InvalidLoRaParametersError(
            f"tx_power must be a number, got {type(tx_power).__name__}"
        )
    if not (2 <= tx_power <= 30):
        raise InvalidLoRaParametersError(
            f"tx_power {tx_power} dBm out of range [2, 30]. "
            f"Typical values: 14 dBm (standard), 20 dBm (high power)"
        )
    if tx_power > 20:
        logger.warning(
            f"TX power {tx_power} dBm is very high. "
            f"Ensure your hardware supports this. Typical max: 20 dBm"
        )
    
    # Frequency
    if not isinstance(frequency, int):
        raise InvalidLoRaParametersError(
            f"frequency must be a number, got {type(frequency).__name__}"
        )
    if not (100 <= frequency <= 1000):
        raise InvalidLoRaParametersError(
            f"frequency {frequency} MHz out of range [200, 1000]. "
            f"Common bands: EU=868, US=915, AS=923, IN=865"
        )
    
    # Frequency band warnings
    if 863 <= frequency <= 870:
        logger.info("Using EU863-870 band (Europe)")
    elif 902 <= frequency <= 928:
        logger.info("Using US902-928 band (North America)")
    elif 915 <= frequency <= 928:
        logger.info("Using AS923 band (Asia)")
    else:
        logger.warning(
            f"Frequency {frequency} MHz is unusual. "
            f"Standard bands: EU=868, US=915, AS=923"
        )

def validate_grid_parameters(grid_spacing_km: float, corridor_width_km: float, 
                            adaptive_grid: bool):
    """Validate grid configuration parameters"""
    # Grid Spacing
    if not isinstance(grid_spacing_km, (int, float)):
        raise ValueError(
            f"grid_spacing_km must be a number, got {type(grid_spacing_km).__name__}"
        )
    if not (0.2 <= grid_spacing_km <= 10):
        raise ValueError(
            f"grid_spacing_km {grid_spacing_km} out of range [0.2, 10]. "
            f"Recommended: 1.0-2.0 km for best results"
        )
    if grid_spacing_km < 0.5:
        logger.warning(
            f"grid_spacing_km {grid_spacing_km} is very small. "
            f"This will create a very dense grid (slow computation)"
        )
    if grid_spacing_km > 5:
        logger.warning(
            f"grid_spacing_km {grid_spacing_km} is very large. "
            f"This may miss optimal paths. Recommended: 1.0-2.0 km"
        )
    
    # Corridor Width
    if not isinstance(corridor_width_km, (int, float)):
        raise ValueError(
            f"corridor_width_km must be a number, got {type(corridor_width_km).__name__}"
        )
    if not (0.5 <= corridor_width_km <= 20):
        raise ValueError(
            f"corridor_width_km {corridor_width_km} out of range [0.5, 20]. "
            f"Recommended: 3.0-6.0 km"
        )
    if corridor_width_km < 2:
        logger.warning(
            f"corridor_width_km {corridor_width_km} is narrow. "
            f"Path may not find good alternatives around obstacles"
        )
    
    # Adaptive Grid
    if not isinstance(adaptive_grid, bool):
        raise ValueError(
            f"adaptive_grid must be True or False, got {type(adaptive_grid).__name__}"
        )

def validate_gee_parameters(gee_workers: int):
    """Validate Google Earth Engine parameters"""
    if not isinstance(gee_workers, int):
        raise ValueError(
            f"gee_workers must be an integer, got {type(gee_workers).__name__}"
        )
    if not (1 <= gee_workers <= 20):
        raise ValueError(
            f"gee_workers {gee_workers} out of range [1, 20]. "
            f"Recommended: 5-10 for best speed/stability"
        )
    if gee_workers > 10:
        logger.warning(
            f"gee_workers {gee_workers} is very high. "
            f"May hit API rate limits. Recommended: 5-10"
        )

def validate_optimization_parameters(max_path_deviation: float, min_pdr_threshold: float,
                                    prefer_water: bool, avoid_buildings: bool,
                                    direct_path_threshold_km: float):
    """Validate optimization preference parameters"""
    # Max Path Deviation
    if not isinstance(max_path_deviation, (int, float)):
        raise ValueError(
            f"max_path_deviation must be a number, got {type(max_path_deviation).__name__}"
        )
    if not (0.0 <= max_path_deviation <= 3.0):
        raise ValueError(
            f"max_path_deviation {max_path_deviation} out of range [0.0, 3.0]. "
            f"0.5 = allow 50% longer path, 1.0 = allow 100% longer (double length). "
            f"Recommended: 0.3-1.0"
        )
    if max_path_deviation < 0.1:
        logger.warning(
            f"max_path_deviation {max_path_deviation} is very strict. "
            f"Path will be nearly straight. May fail to find route."
        )
    if max_path_deviation > 1.5:
        logger.warning(
            f"max_path_deviation {max_path_deviation} is very loose. "
            f"Path may zigzag excessively. Recommended: 0.3-1.0"
        )
    
    # Min PDR Threshold
    if not isinstance(min_pdr_threshold, (int, float)):
        raise ValueError(
            f"min_pdr_threshold must be a number, got {type(min_pdr_threshold).__name__}"
        )
    if not (0.0 <= min_pdr_threshold <= 1.0):
        raise ValueError(
            f"min_pdr_threshold {min_pdr_threshold} out of range [0.0, 1.0]. "
            f"0.3 = 30% minimum PDR, 0.5 = 50% minimum. "
            f"Recommended: 0.2-0.5"
        )
    if min_pdr_threshold < 0.1:
        logger.warning(
            f"min_pdr_threshold {min_pdr_threshold} is very low. "
            f"Path may use poor quality links. Recommended: 0.2-0.5"
        )
    if min_pdr_threshold > 0.6:
        logger.warning(
            f"min_pdr_threshold {min_pdr_threshold} is very high. "
            f"May fail to find route. Recommended: 0.2-0.5"
        )
    
    # Prefer Water
    if not isinstance(prefer_water, bool):
        raise ValueError(
            f"prefer_water must be True or False, got {type(prefer_water).__name__}"
        )
    
    # Avoid Buildings
    if not isinstance(avoid_buildings, bool):
        raise ValueError(
            f"avoid_buildings must be True or False, got {type(avoid_buildings).__name__}"
        )
    
    # Direct Path Threshold
    if not isinstance(direct_path_threshold_km, (int, float)):
        raise ValueError(
            f"direct_path_threshold_km must be a number, got {type(direct_path_threshold_km).__name__}"
        )
    if not (0.1 <= direct_path_threshold_km <= 10.0):
        raise ValueError(
            f"direct_path_threshold_km {direct_path_threshold_km} out of range [0.1, 10.0]. "
            f"1.0 = use direct path for distances < 1 km. "
            f"Recommended: 0.5-2.0"
        )
    if direct_path_threshold_km > 5.0:
        logger.warning(
            f"direct_path_threshold_km {direct_path_threshold_km} is very large. "
            f"System will attempt direct links over long distances. "
            f"This may result in poor quality. Recommended: 0.5-2.0"
        )

def validate_direct_path_parameters(use_grid: bool, num_samples: Optional[int], 
                                   spacing_km: Optional[float]):
    """Validate direct path sampling parameters"""
    if not isinstance(use_grid, bool):
        raise ValueError(f"direct_path_use_grid must be True or False")
    
    if num_samples is not None:
        if not isinstance(num_samples, int):
            raise ValueError(f"direct_path_samples must be an integer")
        if not (2 <= num_samples <= 100):
            raise ValueError(f"direct_path_samples {num_samples} out of range [2, 100]")
    
    if spacing_km is not None:
        if not isinstance(spacing_km, (int, float)):
            raise ValueError(f"direct_path_spacing_km must be a number")
        if not (0.1 <= spacing_km <= 10.0):
            raise ValueError(f"direct_path_spacing_km {spacing_km} out of range [0.1, 10.0]")
    
    # Ensure only one option is specified
    if not use_grid:
        if num_samples is not None and spacing_km is not None:
            logger.warning(
                "Both num_samples and spacing_km specified. "
                "num_samples takes priority."
            )


### LoRa Physics Engine and Data Loading

In [9]:
class LoRaPhysicsEngine:
    """
    FIXED: More realistic PDR calculation with proper RF modeling
    """
    
    def __init__(self):
        self.snr_thresholds = SNR_THRESHOLD
        self.land_cover_k = LAND_COVER_TO_K
    
    def calculate_pdr(self, snr: float, spreading_factor: int, land_cover: int) -> float:
        """
        FIXED: More realistic PDR calculation
        
        Uses sigmoid-like transition instead of simple exponential
        This creates more realistic behavior where buildings significantly degrade PDR
        """
        snr_threshold = self.snr_thresholds.get(spreading_factor, -7.5)
        margin = snr - snr_threshold
        
        # CRITICAL FIX: Below threshold = near-zero PDR (not exactly 0 for numerical stability)
        if margin <= -5:
            return 0.01  # 1% - very poor
        elif margin <= 0:
            # Rapid decay below threshold
            return 0.05 * np.exp(margin)  # 0.01 to 0.05
        
        # Get decay constant (lower for buildings = slower recovery)
        k = self.land_cover_k.get(land_cover, 0.2)
        
        # FIXED: Sigmoid-like recovery (more realistic)
        # Buildings (k=0.08) need MUCH higher SNR margin to achieve good PDR
        # Cropland (k=0.25) achieves good PDR with moderate SNR margin
        pdr = 1.0 / (1.0 + np.exp(-k * (margin - 5)))
        
        return max(0.01, min(0.99, pdr))
    
    def get_sensitivity(self, spreading_factor: int) -> float:
        """Get receiver sensitivity for given SF"""
        sensitivity_map = {
            7: -123, 8: -126, 9: -129, 10: -132, 11: -134, 12: -137
        }
        return sensitivity_map.get(spreading_factor, -123)


# Data Loading
class UnifiedFeatureBuilder:
    """Builds consistent 15-feature vectors for predictions"""
    
    @staticmethod
    def build_feature_vector(point: PathPoint, lora_params: LoRaParameters) -> np.ndarray:
        """Build 15-feature vector for ML prediction"""
        features = np.array([[
            point.elevation,
            point.land_cover,
            point.terrain_penalty,
            point.distance_to_start,
            lora_params.spreading_factor,
            lora_params.frequency,
            lora_params.tx_power,
            point.elevation / 1000.0,
            point.path_built_up_fraction,
            point.path_vegetation_fraction,
            point.path_water_fraction,
            point.path_avg_penalty,
            point.path_elevation_std,
            point.max_terrain_obstruction_m,
            point.path_dominant_land_cover
        ]])
        
        return features

class LoRaDataPreprocessor:
    """Data loading and preprocessing"""
    
    def __init__(self, gee_integration: Optional[BatchGEEIntegration] = None):
        self.scaler = StandardScaler()
        self.gee = gee_integration

    def load_dataset(self, filepath):
        """Load dataset with flexible format handling"""
        try:
            for sep in [',', '|', '\t']:
                try:
                    df = pd.read_csv(filepath, sep=sep)
                    if len(df.columns) > 5:
                        break
                except:
                    continue
            else:
                raise ValueError("Could not determine file format")
            
            column_mapping = {
                'latitude': ['latitude', 'lat'],
                'longitude': ['longitude', 'lon'],
                'elevation': ['elevation', 'altitude', 'elev'],
                'land_cover': ['land_cover', 'land_cover_code', 'landcover'],
                'terrain_penalty': ['terrain_penalty', 'terrain'],
                'RSSI': ['RSSI', 'rssi'],
                'SNR': ['SNR', 'snr'],
                'observed_path_loss': ['observed_path_loss', 'path_loss', 'loss'],
                'spreading_factor': ['spreading_factor', 'sf'],
                'frequency': ['frequency', 'freq'],
                'tx_power': ['tx_power', 'power'],
                'distance_to_start': ['distance_to_start'],
                'path_built_up_fraction': ['path_built_up_fraction'],
                'path_vegetation_fraction': ['path_vegetation_fraction'],
                'path_water_fraction': ['path_water_fraction'],
                'path_avg_penalty': ['path_avg_penalty'],
                'path_elevation_std': ['path_elevation_std'],
                'max_terrain_obstruction_m': ['max_terrain_obstruction_m'],
                'path_dominant_land_cover': ['path_dominant_land_cover']
            }
            
            df_processed = pd.DataFrame()
            for std_col, possible_cols in column_mapping.items():
                for col in possible_cols:
                    if col in df.columns:
                        df_processed[std_col] = df[col]
                        break
                else:
                    if std_col == 'spreading_factor':
                        df_processed[std_col] = 7
                    elif std_col == 'frequency':
                        df_processed[std_col] = 868
                    elif std_col == 'tx_power':
                        df_processed[std_col] = 14
                    elif std_col in ['terrain_penalty', 'path_avg_penalty']:
                        df_processed[std_col] = 0.5
                    elif std_col in ['path_built_up_fraction', 'path_vegetation_fraction', 
                                   'path_water_fraction', 'path_elevation_std', 
                                   'max_terrain_obstruction_m']:
                        df_processed[std_col] = 0.0
                    elif std_col == 'path_dominant_land_cover':
                        df_processed[std_col] = 50
                    else:
                        df_processed[std_col] = 0
            
            return df_processed.dropna(subset=['RSSI', 'SNR'])
            
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            return pd.DataFrame()

    def merge_datasets(self, *datasets):
        """Merge and clean datasets"""
        valid_datasets = [df for df in datasets if not df.empty]
        
        if not valid_datasets:
            raise ValueError("All datasets are empty!")
        
        if len(valid_datasets) == 1:
            df_combined = valid_datasets[0].copy()
        else:
            df_combined = pd.concat(valid_datasets, ignore_index=True)
        
        df_combined = df_combined.dropna(subset=['RSSI', 'SNR'])
        df_combined['elevation_normalized'] = df_combined['elevation'] / 1000
        
        logger.info(f"Combined dataset shape: {df_combined.shape}")
        return df_combined

    def prepare_features(self, df, target_cols=['RSSI', 'SNR', 'observed_path_loss']):
        """Prepare 15-feature dataset for training"""
        feature_cols = [
            'elevation', 'land_cover', 'terrain_penalty',
            'distance_to_start',
            'spreading_factor', 'frequency', 'tx_power',
            'elevation_normalized',
            'path_built_up_fraction',
            'path_vegetation_fraction',
            'path_water_fraction',
            'path_avg_penalty',
            'path_elevation_std',
            'max_terrain_obstruction_m',
            'path_dominant_land_cover'
        ]
        
        missing_cols = [col for col in feature_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Missing feature columns: {missing_cols}")
        
        X = df[feature_cols].values
        y = df[target_cols].values
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        logger.info(f"Training samples: {len(X_train)}")
        logger.info(f"Test samples: {len(X_test)}")
        logger.info(f"Features: {len(feature_cols)}")
        
        return X_train_scaled, X_test_scaled, y_train, y_test, feature_cols


### Google Earth Engine Integration

In [10]:
class RateLimiter:
    """Simple rate limiter for API calls"""
    def __init__(self, calls_per_second=10):
        self.calls_per_second = calls_per_second
        self.last_call = 0
        
    def __enter__(self):
        elapsed = time.time() - self.last_call
        if elapsed < 1.0 / self.calls_per_second:
            time.sleep((1.0 / self.calls_per_second) - elapsed)
        self.last_call = time.time()
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        pass

class BatchGEEIntegration:
    """Robust batch spatial data fetching from Google Earth Engine"""
    
    def __init__(self, config: GEEConfig):
        self.config = config
        self.initialized = False
        self.cache = {}
        self.rate_limiter = RateLimiter(calls_per_second=10)
        self.physics_engine = LoRaPhysicsEngine()
        
        if config.cache_enabled and os.path.exists(config.cache_file):
            try:
                with open(config.cache_file, 'rb') as f:
                    self.cache = pickle.load(f)
                logger.info(f"Loaded {len(self.cache)} cached GEE results")
            except Exception as e:
                logger.warning(f"Could not load cache: {e}")
        
        self._initialize_gee()
    
    def _initialize_gee(self):
        """Initialize GEE with error handling"""
        try:
            project_id = os.getenv('GEE_PROJECT_ID')
            if project_id:
                ee.Initialize(project=project_id)
            else:
                ee.Initialize()
            
            test_point = ee.Geometry.Point([26, 26])
            test_result = ee.Image('USGS/SRTMGL1_003').sample(test_point, scale=30).getInfo()
            self.initialized = True
            logger.info("Google Earth Engine initialized successfully")
            
        except Exception as e:
            logger.error(f"Google Earth Engine initialization failed: {e}")
            raise RuntimeError(f"Cannot initialize GEE: {e}")
    
    def _get_cache_key(self, lat: float, lon: float, data_type: str) -> str:
        """Generate cache key"""
        return f"{data_type}_{lat:.6f}_{lon:.6f}"
    
    def get_elevation(self, lat: float, lon: float) -> float:
        """Fetch elevation from SRTM"""
        cache_key = self._get_cache_key(lat, lon, 'elevation')
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        try:
            with self.rate_limiter:
                point = ee.Geometry.Point([lon, lat])
                srtm = ee.Image('USGS/SRTMGL1_003')
                elevation_dict = srtm.reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=point,
                    scale=30,
                    maxPixels=1
                ).getInfo()
                
                elevation = elevation_dict.get('elevation')
                if elevation is not None:
                    elevation = float(elevation)
                    self.cache[cache_key] = elevation
                    return elevation
                else:
                    return 0.0
                    
        except Exception as e:
            logger.error(f"Failed to get elevation for ({lat}, {lon}): {e}")
            return 0.0
    
    def get_land_cover(self, lat: float, lon: float) -> Tuple[int, float]:
        """Fetch land cover from ESA WorldCover"""
        cache_key = self._get_cache_key(lat, lon, 'landcover')
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        try:
            with self.rate_limiter:
                point = ee.Geometry.Point([lon, lat])
                worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
                lc_dict = worldcover.reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=point,
                    scale=10,
                    maxPixels=1
                ).getInfo()
                
                land_cover = lc_dict.get('Map')
                if land_cover is not None:
                    land_cover_code = int(land_cover)
                    terrain_penalty = PENALTY_MAP.get(land_cover_code, 0.5)
                    result = (land_cover_code, terrain_penalty)
                    self.cache[cache_key] = result
                    return result
                else:
                    return 50, 0.5  # Default to built-up if no data
                    
        except Exception as e:
            logger.error(f"Failed to get land cover for ({lat}, {lon}): {e}")
            return 50, 0.5
    
    def get_spatial_features(self, lat: float, lon: float) -> Dict:
        """Fetch all spatial features for a single location"""
        cache_key = self._get_cache_key(lat, lon, 'spatial')
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        elevation = self.get_elevation(lat, lon)
        land_cover, terrain_penalty = self.get_land_cover(lat, lon)
        
        result = {
            'elevation': elevation,
            'land_cover': land_cover,
            'terrain_penalty': terrain_penalty
        }
        
        self.cache[cache_key] = result
        return result
    
    def get_path_spatial_features(self, lat1: float, lon1: float, 
                              lat2: float, lon2: float) -> Dict:
        """Compute path-based spatial features between two points"""
        
        # Check cache first (with 4 decimal precision for better hit rate)
        cache_key = f"path_{lat1:.4f}_{lon1:.4f}_{lat2:.4f}_{lon2:.4f}"
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        num_samples = self.config.path_spatial_samples
        
        lats = np.linspace(lat1, lat2, num_samples)
        lons = np.linspace(lon1, lon2, num_samples)
        
        built_up = veg = water = 0
        penalties = []
        elevations = []
        land_covers = []
        
        # Fetch all points in the path
        for lat, lon in zip(lats, lons):
            try:
                lc, penalty = self.get_land_cover(lat, lon)
                elev = self.get_elevation(lat, lon)
                
                if lc == 50:
                    built_up += 1
                elif lc in {10, 20, 90, 95}:
                    veg += 1
                elif lc == 80:
                    water += 1
                
                penalties.append(penalty)
                elevations.append(elev)
                land_covers.append(lc)
                
            except Exception as e:
                logger.debug(f"Skipping point ({lat:.4f}, {lon:.4f}): {e}")
                continue
        
        total = len(elevations) if elevations else 1
        
        result = {
            'path_built_up_fraction': built_up / total,
            'path_vegetation_fraction': veg / total,
            'path_water_fraction': water / total,
            'path_avg_penalty': np.mean(penalties) if penalties else 0.5,
            'path_elevation_std': np.std(elevations) if len(elevations) > 1 else 0.0,
            'max_terrain_obstruction_m': float(np.max(elevations) - np.min(elevations)) if elevations else 0.0,
            'path_dominant_land_cover': int(np.median(land_covers)) if land_covers else 50
        }
        
        # Cache the result
        self.cache[cache_key] = result
        
        return result
    
    def batch_get_path_spatial_features(self, path_pairs: List[Tuple[Tuple[float, float], Tuple[float, float]]]) -> List[Dict]:
        """
        Batch fetch path spatial features for multiple path segments
        Uses parallel processing and caching for efficiency
        
        Args:
            path_pairs: List of ((lat1, lon1), (lat2, lon2)) tuples
        
        Returns:
            List of path feature dictionaries
        """
        logger.info(f"Batch fetching path features for {len(path_pairs)} segments...")
        
        # Deduplicate path pairs
        unique_pairs = list(set(path_pairs))
        pair_to_indices = {pair: [] for pair in unique_pairs}
        for idx, pair in enumerate(path_pairs):
            pair_to_indices[pair].append(idx)
        
        results_map = {}
        
        # Check cache first
        uncached_pairs = []
        for pair in unique_pairs:
            (lat1, lon1), (lat2, lon2) = pair
            cache_key = f"path_{lat1:.4f}_{lon1:.4f}_{lat2:.4f}_{lon2:.4f}"
            
            if cache_key in self.cache:
                results_map[pair] = self.cache[cache_key]
            else:
                uncached_pairs.append(pair)
        
        logger.info(f"  Cached: {len(unique_pairs) - len(uncached_pairs)}, Need to fetch: {len(uncached_pairs)}")
        
        # Fetch uncached paths in parallel
        if uncached_pairs:
            with ThreadPoolExecutor(max_workers=self.config.workers) as executor:
                future_to_pair = {
                    executor.submit(self.get_path_spatial_features, pair[0][0], pair[0][1], pair[1][0], pair[1][1]): pair
                    for pair in uncached_pairs
                }
                
                with tqdm(total=len(uncached_pairs), desc="Fetching Path Features", unit="paths") as pbar:
                    for future in as_completed(future_to_pair):
                        pair = future_to_pair[future]
                        try:
                            result = future.result()
                            results_map[pair] = result
                            
                            # Cache it
                            (lat1, lon1), (lat2, lon2) = pair
                            cache_key = f"path_{lat1:.4f}_{lon1:.4f}_{lat2:.4f}_{lon2:.4f}"
                            self.cache[cache_key] = result
                            
                        except Exception as e:
                            logger.warning(f"Failed to fetch path features for {pair}: {e}")
                            
                            # SMART FALLBACK: Use endpoint grid point data
                            (lat1, lon1), (lat2, lon2) = pair
                            try:
                                # Try to at least get endpoint data
                                start_spatial = self.get_spatial_features(lat1, lon1)
                                end_spatial = self.get_spatial_features(lat2, lon2)
                                
                                # Interpolate
                                results_map[pair] = {
                                    'path_built_up_fraction': 0.5 if (start_spatial['land_cover'] == 50 or end_spatial['land_cover'] == 50) else 0.0,
                                    'path_vegetation_fraction': 0.5 if (start_spatial['land_cover'] in {10,20,90,95} or end_spatial['land_cover'] in {10,20,90,95}) else 0.0,
                                    'path_water_fraction': 0.5 if (start_spatial['land_cover'] == 80 or end_spatial['land_cover'] == 80) else 0.0,
                                    'path_avg_penalty': (start_spatial['terrain_penalty'] + end_spatial['terrain_penalty']) / 2.0,
                                    'path_elevation_std': abs(end_spatial['elevation'] - start_spatial['elevation']) / 2.0,
                                    'max_terrain_obstruction_m': max(start_spatial['elevation'], end_spatial['elevation']),
                                    'path_dominant_land_cover': end_spatial['land_cover']
                                }
                            except:
                                # Ultimate fallback: use safe defaults
                                results_map[pair] = {
                                    'path_built_up_fraction': 0.2,
                                    'path_vegetation_fraction': 0.3,
                                    'path_water_fraction': 0.1,
                                    'path_avg_penalty': 0.5,
                                    'path_elevation_std': 50.0,
                                    'max_terrain_obstruction_m': 100.0,
                                    'path_dominant_land_cover': 50
                                }
                        pbar.update(1)
        
        # Map back to original order with duplicates
        results = [results_map[path_pairs[i]] for i in range(len(path_pairs))]
        
        # Save cache
        if self.config.cache_enabled and uncached_pairs:
            try:
                with open(self.config.cache_file, 'wb') as f:
                    pickle.dump(self.cache, f)
            except Exception as e:
                logger.warning(f"Could not save cache: {e}")
        
        return results

    def batch_fetch_spatial_features(self, coordinates: List[Tuple[float, float]]) -> List[Dict]:
        """Fetch spatial features for multiple locations using parallel workers"""
        total = len(coordinates)
        logger.info(f"Fetching spatial features for {total} locations with {self.config.workers} workers...")
        
        results = [None] * total
        
        with ThreadPoolExecutor(max_workers=self.config.workers) as executor:
            future_to_idx = {
                executor.submit(self.get_spatial_features, lat, lon): idx
                for idx, (lat, lon) in enumerate(coordinates)
            }
            
            with tqdm(total=total, desc="GEE Batch Fetch", unit="points") as pbar:
                for future in as_completed(future_to_idx):
                    idx = future_to_idx[future]
                    try:
                        result = future.result()
                        result['latitude'] = coordinates[idx][0]
                        result['longitude'] = coordinates[idx][1]
                        results[idx] = result
                    except Exception as e:
                        logger.warning(f"Failed to fetch data for point {idx}: {e}")
                        results[idx] = {
                            'latitude': coordinates[idx][0],
                            'longitude': coordinates[idx][1],
                            'elevation': 0.0,
                            'land_cover': 50,
                            'terrain_penalty': 0.5
                        }
                    pbar.update(1)
        
        if self.config.cache_enabled:
            try:
                with open(self.config.cache_file, 'wb') as f:
                    pickle.dump(self.cache, f)
                logger.info(f"Saved {len(self.cache)} GEE results to cache")
            except Exception as e:
                logger.warning(f"Could not save cache: {e}")
        
        return results


### PyTorch Neural Network

In [11]:
class LoRaDataset(Dataset):
    """PyTorch dataset for LoRa data"""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LoRaNeuralNetwork(nn.Module):
    """Neural network with ALL features implemented"""
    def __init__(self, input_size, output_size, config=None):
        super(LoRaNeuralNetwork, self).__init__()
        
        default_config = {
            'hidden_sizes': [256, 128, 64, 32],
            'dropout_rate': 0.3,
            'dropout_rates': None,
            'activation': 'relu',
            'leaky_alpha': 0.2,
            'elu_alpha': 1.0,
            'normalization': 'batch_norm',
            'norm_position': 'before_activation',
            'num_groups': 8,
            'use_residual': False,
            'residual_frequency': 2,
            'init_method': 'xavier_uniform'
        }
        if config:
            default_config.update(config)
        self.config = default_config
        
        self.use_residual = self.config['use_residual']
        self.residual_frequency = self.config.get('residual_frequency', 2)
        
        layers = []
        self.residual_layers = nn.ModuleList()
        prev_size = input_size
        
        for i, hidden_size in enumerate(self.config['hidden_sizes']):
            # Linear layer
            linear = nn.Linear(prev_size, hidden_size)
            self._initialize_weights(linear, self.config['init_method'])
            layers.append(linear)
            
            # Normalization (before or after activation)
            if self.config['norm_position'] == 'before_activation':
                layers.append(self._get_normalization(hidden_size))
            
            # Activation
            layers.append(self._get_activation())
            
            # Normalization (after activation)
            if self.config['norm_position'] == 'after_activation':
                layers.append(self._get_normalization(hidden_size))
            
            # Dropout
            dropout_rate = self.config['dropout_rates'][i] if self.config['dropout_rates'] else self.config['dropout_rate']
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            
            # Residual connections
            if self.use_residual and i > 0 and (i % self.residual_frequency == 0):
                if prev_size != hidden_size:
                    self.residual_layers.append(nn.Linear(prev_size, hidden_size))
                else:
                    self.residual_layers.append(nn.Identity())
            
            prev_size = hidden_size
        
        # Output layer
        output_layer = nn.Linear(prev_size, output_size)
        self._initialize_weights(output_layer, self.config['init_method'])
        layers.append(output_layer)
        
        self.network = nn.Sequential(*layers)
        self.residual_counter = 0

    def _get_activation(self):
        """Get activation function"""
        act = self.config['activation']
        if act == 'relu':
            return nn.ReLU()
        elif act == 'leaky_relu':
            return nn.LeakyReLU(self.config['leaky_alpha'])
        elif act == 'prelu':
            return nn.PReLU()
        elif act == 'elu':
            return nn.ELU(self.config['elu_alpha'])
        elif act == 'selu':
            return nn.SELU()
        elif act == 'gelu':
            return nn.GELU()
        elif act == 'swish':
            return nn.SiLU()  # Swish = SiLU in PyTorch
        elif act == 'mish':
            return nn.Mish()
        else:
            return nn.ReLU()
    
    def _get_normalization(self, num_features):
        """Get normalization layer"""
        norm = self.config['normalization']
        if norm == 'batch_norm':
            return nn.BatchNorm1d(num_features)
        elif norm == 'layer_norm':
            return nn.LayerNorm(num_features)
        elif norm == 'instance_norm':
            return nn.InstanceNorm1d(num_features, affine=True)
        elif norm == 'group_norm':
            num_groups = min(self.config['num_groups'], num_features)
            return nn.GroupNorm(num_groups, num_features)
        elif norm == 'none':
            return nn.Identity()
        else:
            return nn.BatchNorm1d(num_features)
    
    def _initialize_weights(self, layer, method):
        """Initialize layer weights"""
        if not isinstance(layer, nn.Linear):
            return
        
        if method == 'xavier_uniform':
            nn.init.xavier_uniform_(layer.weight)
        elif method == 'xavier_normal':
            nn.init.xavier_normal_(layer.weight)
        elif method == 'kaiming_uniform':
            nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
        elif method == 'kaiming_normal':
            nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
        elif method == 'orthogonal':
            nn.init.orthogonal_(layer.weight)
        
        if layer.bias is not None:
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.network(x)

    def predict(self, X):
        """Make predictions"""
        self.eval()
        with torch.no_grad():
            if isinstance(X, np.ndarray):
                X = torch.FloatTensor(X)
            X = X.to(next(self.parameters()).device)
            predictions = self.forward(X).cpu().numpy()
        return predictions

class NeuralNetworkTrainer:
    """Neural network trainer with ALL features implemented"""
    def __init__(self, input_size, output_size, device, config=None):
        self.device = device
        self.config = config or {}
        
        model_config = self.config.get('model', {})
        self.model = LoRaNeuralNetwork(input_size, output_size, model_config).to(device)
        
        self.criterion = nn.MSELoss()
        
        # L1 regularization
        self.l1_lambda = self.config.get('l1_lambda', 0.0)
        
        # Build optimizer with all options
        self.optimizer = self._build_optimizer()
        
        # Build scheduler
        self.scheduler = self._build_scheduler()
        
        self.early_stopping_patience = self.config.get('early_stopping_patience', 20)
        self.early_stopping_counter = 0
        self.best_val_loss = float('inf')
        self.train_losses = []
        self.val_losses = []
        self.best_model_state = None
        
        # Gradient accumulation
        self.accumulation_steps = self.config.get('accumulation_steps', 1)
        
        # Mixed precision
        self.use_mixed_precision = self.config.get('use_mixed_precision', False)
        self.scaler = torch.cuda.amp.GradScaler() if self.use_mixed_precision else None
    
    def _build_optimizer(self):
        """Build optimizer based on config"""
        opt_name = self.config.get('optimizer_name', 'adam')
        lr = self.config.get('learning_rate', 0.001)
        weight_decay = self.config.get('weight_decay', 1e-5)
        
        if opt_name == 'adam':
            return optim.Adam(
                self.model.parameters(),
                lr=lr,
                betas=(self.config.get('beta1', 0.9), self.config.get('beta2', 0.999)),
                eps=self.config.get('epsilon', 1e-8),
                weight_decay=weight_decay
            )
        elif opt_name == 'adamw':
            return optim.AdamW(
                self.model.parameters(),
                lr=lr,
                betas=(self.config.get('beta1', 0.9), self.config.get('beta2', 0.999)),
                eps=self.config.get('epsilon', 1e-8),
                weight_decay=weight_decay
            )
        elif opt_name == 'radam':
            return optim.RAdam(
                self.model.parameters(),
                lr=lr,
                betas=(self.config.get('beta1', 0.9), self.config.get('beta2', 0.999)),
                eps=self.config.get('epsilon', 1e-8),
                weight_decay=weight_decay
            )
        elif opt_name == 'nadam':
            return optim.NAdam(
                self.model.parameters(),
                lr=lr,
                betas=(self.config.get('beta1', 0.9), self.config.get('beta2', 0.999)),
                eps=self.config.get('epsilon', 1e-8),
                weight_decay=weight_decay
            )
        elif opt_name == 'adamax':
            return optim.Adamax(
                self.model.parameters(),
                lr=lr,
                betas=(self.config.get('beta1', 0.9), self.config.get('beta2', 0.999)),
                eps=self.config.get('epsilon', 1e-8),
                weight_decay=weight_decay
            )
        elif opt_name == 'sgd':
            return optim.SGD(
                self.model.parameters(),
                lr=lr,
                momentum=self.config.get('momentum', 0.9),
                weight_decay=weight_decay,
                nesterov=self.config.get('nesterov', False)
            )
        elif opt_name == 'rmsprop':
            return optim.RMSprop(
                self.model.parameters(),
                lr=lr,
                alpha=self.config.get('rmsprop_alpha', 0.99),
                momentum=self.config.get('momentum', 0.0),
                weight_decay=weight_decay
            )
        else:
            return optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
    
    def _build_scheduler(self):
        """Build learning rate scheduler"""
        if not self.config.get('use_scheduler', True):
            return None
        
        scheduler_type = self.config.get('scheduler_type', 'plateau')
        
        if scheduler_type == 'step':
            return optim.lr_scheduler.StepLR(
                self.optimizer,
                step_size=self.config.get('step_size', 10),
                gamma=self.config.get('gamma', 0.5)
            )
        elif scheduler_type == 'exponential':
            return optim.lr_scheduler.ExponentialLR(
                self.optimizer,
                gamma=self.config.get('gamma', 0.95)
            )
        elif scheduler_type == 'cosine':
            return optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer,
                T_max=self.config.get('T_max', 50),
                eta_min=self.config.get('eta_min', 1e-6)
            )
        elif scheduler_type == 'plateau':
            return optim.lr_scheduler.ReduceLROnPlateau(
                self.optimizer,
                mode='min',
                patience=self.config.get('scheduler_patience', 10),
                factor=self.config.get('scheduler_factor', 0.5),
                verbose=True
            )
        elif scheduler_type == 'cyclic':
            return optim.lr_scheduler.CyclicLR(
                self.optimizer,
                base_lr=self.config.get('learning_rate', 0.001) / 10,
                max_lr=self.config.get('learning_rate', 0.001) * 10,
                step_size_up=self.config.get('step_size_up', 10),
                mode='triangular2'
            )
        elif scheduler_type == 'onecycle':
            return None  # Will be set in train() with actual steps
        else:
            return optim.lr_scheduler.ReduceLROnPlateau(
                self.optimizer, mode='min', patience=10, factor=0.5
            )

    def train(self, train_loader, val_loader, epochs=None):
        """Train the neural network with ALL features"""
        epochs = epochs or self.config.get('epochs', 100)
        
        # Create OneCycleLR if needed
        if self.config.get('scheduler_type') == 'onecycle' and self.config.get('use_scheduler'):
            total_steps = epochs * len(train_loader)
            self.scheduler = optim.lr_scheduler.OneCycleLR(
                self.optimizer,
                max_lr=self.config.get('learning_rate', 0.001) * self.config.get('max_lr_multiplier', 10),
                total_steps=total_steps,
                pct_start=self.config.get('pct_start', 0.3)
            )
        
        logger.info(f"Training Neural Network on {self.device}...")
        
        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_loss = 0
            train_steps = 0
            
            for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                
                # Mixed precision training
                if self.use_mixed_precision:
                    with torch.cuda.amp.autocast():
                        outputs = self.model(X_batch)
                        loss = self.criterion(outputs, y_batch)
                        
                        # L1 regularization
                        if self.l1_lambda > 0:
                            l1_norm = sum(p.abs().sum() for p in self.model.parameters())
                            loss = loss + self.l1_lambda * l1_norm
                        
                        loss = loss / self.accumulation_steps
                    
                    self.scaler.scale(loss).backward()
                    
                    if (batch_idx + 1) % self.accumulation_steps == 0:
                        # Gradient clipping
                        if self.config.get('gradient_clip', 0) > 0:
                            self.scaler.unscale_(self.optimizer)
                            if self.config.get('gradient_clip_type') == 'value':
                                nn.utils.clip_grad_value_(self.model.parameters(), self.config['gradient_clip'])
                            else:
                                nn.utils.clip_grad_norm_(self.model.parameters(), self.config['gradient_clip'])
                        
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                        self.optimizer.zero_grad()
                else:
                    outputs = self.model(X_batch)
                    loss = self.criterion(outputs, y_batch)
                    
                    # L1 regularization
                    if self.l1_lambda > 0:
                        l1_norm = sum(p.abs().sum() for p in self.model.parameters())
                        loss = loss + self.l1_lambda * l1_norm
                    
                    loss = loss / self.accumulation_steps
                    loss.backward()
                    
                    if (batch_idx + 1) % self.accumulation_steps == 0:
                        # Gradient clipping
                        if self.config.get('gradient_clip', 0) > 0:
                            if self.config.get('gradient_clip_type') == 'value':
                                nn.utils.clip_grad_value_(self.model.parameters(), self.config['gradient_clip'])
                            else:
                                nn.utils.clip_grad_norm_(self.model.parameters(), self.config['gradient_clip'])
                        
                        self.optimizer.step()
                        self.optimizer.zero_grad()
                
                train_loss += loss.item() * self.accumulation_steps
                train_steps += 1
                
                # Step scheduler for batch-level schedulers
                if self.scheduler and self.config.get('scheduler_type') in ['cyclic', 'onecycle']:
                    self.scheduler.step()
            
            train_loss /= train_steps
            self.train_losses.append(train_loss)
            
            # Validation phase
            self.model.eval()
            val_loss = 0
            val_steps = 0
            
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                    outputs = self.model(X_batch)
                    loss = self.criterion(outputs, y_batch)
                    val_loss += loss.item()
                    val_steps += 1
            
            val_loss /= val_steps
            self.val_losses.append(val_loss)
            
            # Step scheduler for epoch-level schedulers
            if self.scheduler:
                scheduler_type = self.config.get('scheduler_type', 'plateau')
                
                # Don't step batch-level schedulers here
                if scheduler_type in ['cyclic', 'onecycle']:
                    pass  # Already stepped in training loop
                # ReduceLROnPlateau needs metric
                elif scheduler_type == 'plateau':
                    self.scheduler.step(val_loss)
                # All other schedulers don't need metric
                else:
                    self.scheduler.step()
            
            # Early stopping
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.early_stopping_counter = 0
                self.best_model_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
            else:
                self.early_stopping_counter += 1
                if self.early_stopping_counter >= self.early_stopping_patience:
                    logger.info(f"Early stopping at epoch {epoch+1}")
                    if self.best_model_state:
                        self.model.load_state_dict(self.best_model_state)
                    break
            
            if (epoch + 1) % 10 == 0:
                logger.info(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        
        # Load best model
        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
        
        logger.info("Neural Network training completed!")

    def predict(self, X):
        """Make predictions"""
        return self.model.predict(X)


### Hyperparameter Tuning for Random Forest, XGBoost, and Neural Network

In [12]:
class RandomForestTuner:
    """Hyperparameter tuning for Random Forest"""
    def __init__(self, X_train, y_train, n_iter=50, cv_folds=5):
        self.X_train = X_train
        self.y_train = y_train
        self.n_iter = n_iter
        self.cv_folds = cv_folds
    
    def tune(self):
        """Run Random Forest hyperparameter tuning"""
        logger.info(f"Running Random Forest tuning ({self.n_iter} iterations)...")
        
        rf = RandomForestRegressor(random_state=42, n_jobs=-1)
        
        # FIXED: Removed max_samples and oob_score from param_dist
        param_dist = {
            'n_estimators': randint(50, 500),
            'max_depth': [None] + list(range(5, 50, 5)),
            'min_samples_split': randint(2, 20),
            'min_samples_leaf': randint(1, 10),
            'min_weight_fraction_leaf': uniform(0.0, 0.1),
            'max_features': ['sqrt', 'log2', None, 0.3, 0.5, 0.7],
            'max_leaf_nodes': [None] + list(range(20, 200, 20)),
            'min_impurity_decrease': uniform(0.0, 0.1),
            'bootstrap': [True, False],
            'ccp_alpha': uniform(0.0, 0.05),
            'warm_start': [False],  # Keep False for CV
            'random_state': [42]
        }
        
        random_search = RandomizedSearchCV(
            rf,
            param_distributions=param_dist,
            n_iter=self.n_iter,
            cv=self.cv_folds,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            random_state=42,
            verbose=1,
            error_score='raise'
        )
        
        logger.info("  Starting Random Forest hyperparameter search...")
        random_search.fit(self.X_train, self.y_train[:, 0])
        
        best_params = random_search.best_params_
        best_score = random_search.best_score_
        
        logger.info(f"  Best CV Score (MSE): {-best_score:.6f}")
        logger.info("  Best hyperparameters:")
        for key, value in best_params.items():
            logger.info(f"    {key}: {value}")
        
        # FIXED: Only tune max_samples if bootstrap=True
        if best_params.get('bootstrap', False):
            logger.info("  Fine-tuning max_samples (bootstrap=True)...")
            best_max_samples = None
            best_subsample_score = best_score
            
            for max_samp in [0.5, 0.6, 0.7, 0.8, 0.9, None]:
                try:
                    rf_temp = RandomForestRegressor(
                        **best_params,
                        max_samples=max_samp,
                        n_jobs=-1
                    )
                    scores = cross_val_score(
                        rf_temp, self.X_train, self.y_train[:, 0],
                        cv=self.cv_folds,
                        scoring='neg_mean_squared_error',
                        n_jobs=-1
                    )
                    mean_score = scores.mean()
                    
                    if mean_score > best_subsample_score:
                        best_subsample_score = mean_score
                        best_max_samples = max_samp
                        logger.info(f"    max_samples={max_samp}: {-mean_score:.6f} ")
                except Exception as e:
                    logger.warning(f"    max_samples={max_samp}: Failed")
                    continue
            
            if best_max_samples is not None:
                best_params['max_samples'] = best_max_samples
                logger.info(f"  Selected max_samples: {best_max_samples}")
            
            # FIXED: Add oob_score only if bootstrap=True
            best_params['oob_score'] = True
        else:
            logger.info("  Skipping max_samples/oob_score (bootstrap=False)")
            best_params['oob_score'] = False
        
        return best_params

class XGBoostTuner:
    """Hyperparameter tuning for XGBoost"""
    
    def __init__(self, X_train, y_train, n_iter=50, cv_folds=5):
        self.X_train = X_train
        self.y_train = y_train
        self.n_iter = n_iter
        self.cv_folds = cv_folds
    
    def tune(self):
        """Run XGBoost hyperparameter tuning"""
        logger.info(f"Running XGBoost tuning ({self.n_iter} iterations)...")
        
        # FIXED: Separate param_dist for tree-based boosters only
        param_dist = {
            # Core parameters
            'n_estimators': randint(50, 500),
            'learning_rate': uniform(0.01, 0.3),
            'max_depth': randint(3, 12),
            'min_child_weight': randint(1, 10),
            'gamma': uniform(0.0, 0.5),
            
            # Sampling
            'subsample': uniform(0.5, 0.5),
            'colsample_bytree': uniform(0.5, 0.5),
            'colsample_bylevel': uniform(0.5, 0.5),
            'colsample_bynode': uniform(0.5, 0.5),
            
            # Regularization
            'reg_alpha': uniform(0.0, 1.0),
            'reg_lambda': uniform(0.5, 1.5),
            
            # Tree method - FIXED: Only valid methods
            'tree_method': ['auto', 'hist'],
            
            # FIXED: Only gbtree booster (removed gblinear and dart)
            'booster': ['gbtree'],
            
            # Objective
            'objective': ['reg:squarederror'],
            
            # Growth policy
            'grow_policy': ['depthwise', 'lossguide'],
            
            # Max leaves (for lossguide)
            'max_leaves': randint(0, 64),
            
            # Max bin
            'max_bin': randint(128, 512),
            
            # Other
            'random_state': [42],
            'verbosity': [0],
            'n_jobs': [-1]
        }
        
        xgb_model = xgb.XGBRegressor()
        
        random_search = RandomizedSearchCV(
            xgb_model,
            param_distributions=param_dist,
            n_iter=self.n_iter,
            cv=self.cv_folds,
            scoring='neg_mean_squared_error',
            random_state=42,
            n_jobs=-1,
            verbose=1,
            return_train_score=True,
            error_score='raise'
        )
        
        logger.info("  Starting XGBoost hyperparameter search...")
        random_search.fit(self.X_train, self.y_train[:, 0])
        
        best_params = random_search.best_params_
        best_score = random_search.best_score_
        
        logger.info(f"  Best CV Score (MSE): {-best_score:.6f}")
        logger.info("  Best hyperparameters:")
        for key, value in best_params.items():
            logger.info(f"    {key}: {value}")
        
        return best_params

class NeuralNetworkTuner:
    """Comprehensive hyperparameter tuning"""
    def __init__(self, X_train, y_train, X_val, y_val, device, n_trials=50):
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.device = device
        self.n_trials = n_trials
        
        if not OPTUNA_AVAILABLE:
            raise ImportError("Optuna required for tuning. Install: pip install optuna")

    def objective(self, trial):
        """Optuna objective function"""
        
        # Architecture
        n_layers = trial.suggest_int('n_layers', 2, 6)
        hidden_size_base = trial.suggest_categorical('hidden_size_base', [64, 128, 256, 512])
        decay_strategy = trial.suggest_categorical('decay_strategy', ['exponential', 'linear', 'constant'])
        
        if decay_strategy == 'exponential':
            hidden_sizes = [hidden_size_base // (2**i) for i in range(n_layers)]
        elif decay_strategy == 'linear':
            hidden_sizes = [int(hidden_size_base * (1 - i/(n_layers+1))) for i in range(n_layers)]
        else:
            hidden_sizes = [hidden_size_base] * n_layers
        
        hidden_sizes = [max(32, s) for s in hidden_sizes]
        
        # Regularization
        dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.6)
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
        
        # Activation
        activation = trial.suggest_categorical('activation', 
            ['relu', 'leaky_relu', 'elu', 'gelu'])
        
        # Normalization
        normalization = trial.suggest_categorical('normalization', 
            ['batch_norm', 'layer_norm', 'none'])
        
        # Optimizer
        optimizer_name = trial.suggest_categorical('optimizer_name', 
            ['adam', 'adamw', 'sgd'])
        learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
        
        # Training
        batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])
        gradient_clip = trial.suggest_float('gradient_clip', 0.5, 5.0)
        early_stopping_patience = trial.suggest_int('early_stopping_patience', 10, 30)
        
        # Build configs
        model_config = {
            'hidden_sizes': hidden_sizes,
            'dropout_rate': dropout_rate,
            'activation': activation,
            'normalization': normalization,
            'init_method': 'xavier_uniform'
        }
        
        training_config = {
            'learning_rate': learning_rate,
            'weight_decay': weight_decay,
            'optimizer_name': optimizer_name,
            'epochs': 100,
            'early_stopping_patience': early_stopping_patience,
            'gradient_clip': gradient_clip,
            'model': model_config
        }
        
        # Create datasets
        train_dataset = LoRaDataset(self.X_train, self.y_train)
        val_dataset = LoRaDataset(self.X_val, self.y_val)
        
        # Ensure batch size >= 2 for normalization
        effective_batch_size = max(2, batch_size)
        
        train_size = len(train_dataset)
        val_size = len(val_dataset)
        
        if train_size < effective_batch_size * 2:
            effective_batch_size = max(2, train_size // 3)
        if val_size < effective_batch_size * 2:
            effective_batch_size = max(2, min(effective_batch_size, val_size // 3))
        
        train_drop_last = (train_size > effective_batch_size * 3)
        val_drop_last = (val_size > effective_batch_size * 3)
        
        try:
            train_loader = DataLoader(
                train_dataset,
                batch_size=effective_batch_size,
                shuffle=True,
                num_workers=0,
                pin_memory=True if torch.cuda.is_available() else False,
                drop_last=train_drop_last
            )
            val_loader = DataLoader(
                val_dataset,
                batch_size=effective_batch_size,
                shuffle=False,
                num_workers=0,
                pin_memory=True if torch.cuda.is_available() else False,
                drop_last=val_drop_last
            )
            
            if len(train_loader) == 0 or len(val_loader) == 0:
                raise optuna.exceptions.TrialPruned()
            
            trainer = NeuralNetworkTrainer(
                input_size=self.X_train.shape[1],
                output_size=self.y_train.shape[1],
                device=self.device,
                config=training_config
            )
            
            trainer.train(train_loader, val_loader)
            
            return trainer.best_val_loss
            
        except Exception as e:
            logger.warning(f"Trial {trial.number}: {str(e)[:50]}")
            raise optuna.exceptions.TrialPruned()

    def tune(self):
        """Run hyperparameter tuning"""
        logger.info(f"Running Neural Network tuning ({self.n_trials} trials)...")
        
        sampler = TPESampler(seed=42, n_startup_trials=10)
        pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=10)
        
        study = optuna.create_study(
            direction='minimize',
            sampler=sampler,
            pruner=pruner
        )
        
        try:
            study.optimize(
                self.objective,
                n_trials=self.n_trials,
                show_progress_bar=True,
                catch=(RuntimeError, ValueError, Exception)
            )
        except KeyboardInterrupt:
            logger.info("Optimization interrupted")
        
        completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        
        if len(completed_trials) == 0:
            logger.warning("No trials completed! Using defaults")
            return {
                'hidden_sizes': [256, 128, 64],
                'dropout_rate': 0.3,
                'activation': 'relu',
                'normalization': 'batch_norm',
                'batch_size': 32,
                'learning_rate': 0.001,
                'weight_decay': 1e-5,
                'optimizer_name': 'adam',
                'gradient_clip': 1.0,
                'early_stopping_patience': 20
            }
        
        logger.info(f"  Best trial: {study.best_trial.number}")
        logger.info(f"  Best loss: {study.best_value:.6f}")
        logger.info(f"  Completed: {len(completed_trials)}/{len(study.trials)}")
        
        best = study.best_params
        n_layers = best['n_layers']
        
        if best['decay_strategy'] == 'exponential':
            hidden_sizes = [best['hidden_size_base'] // (2**i) for i in range(n_layers)]
        elif best['decay_strategy'] == 'linear':
            hidden_sizes = [int(best['hidden_size_base'] * (1 - i/(n_layers+1))) for i in range(n_layers)]
        else:
            hidden_sizes = [best['hidden_size_base']] * n_layers
        
        hidden_sizes = [max(32, s) for s in hidden_sizes]
        
        return {
            'hidden_sizes': hidden_sizes,
            'dropout_rate': best['dropout_rate'],
            'activation': best['activation'],
            'normalization': best['normalization'],
            'batch_size': best['batch_size'],
            'learning_rate': best['learning_rate'],
            'weight_decay': best['weight_decay'],
            'optimizer_name': best.get('optimizer_name', 'adam'),
            'gradient_clip': best['gradient_clip'],
            'early_stopping_patience': best['early_stopping_patience']
        }

    def _log_callback(self, study, trial):
        """Log callback"""
        if trial.number % 5 == 0 and trial.state == optuna.trial.TrialState.COMPLETE:
            logger.info(f"  Trial {trial.number}: loss={trial.value:.6f}")


### Random Forest, XGBoost, and Ensemble Models, with Model Selector

In [13]:
class RandomForestModel:
    """Random Forest model"""
    def __init__(self, **kwargs):
        default_params = {
            'n_estimators': 100,
            'max_depth': None,
            'min_samples_split': 2,
            'min_samples_leaf': 1,
            'random_state': 42,
            'n_jobs': -1
        }
        default_params.update(kwargs)
        
        self.models = {
            'RSSI': RandomForestRegressor(**default_params),
            'SNR': RandomForestRegressor(**default_params),
            'path_loss': RandomForestRegressor(**default_params)
        }

    def train(self, X_train, y_train):
        """Train all models"""
        logger.info("Training Random Forest models...")
        for i, (name, model) in enumerate(self.models.items()):
            model.fit(X_train, y_train[:, i])
        logger.info("  Random Forest training completed!")

    def predict(self, X):
        """Make predictions"""
        predictions = np.zeros((X.shape[0], len(self.models)))
        for i, model in enumerate(self.models.values()):
            predictions[:, i] = model.predict(X)
        return predictions

    def evaluate(self, X, y):
        """Evaluate model and return detailed metrics"""
        from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
        
        y_pred = self.predict(X)
        metrics = {}
        
        for i, name in enumerate(['RSSI', 'SNR', 'path_loss']):
            metrics[name] = {
                'mse': mean_squared_error(y[:, i], y_pred[:, i]),
                'rmse': np.sqrt(mean_squared_error(y[:, i], y_pred[:, i])),
                'mae': mean_absolute_error(y[:, i], y_pred[:, i]),
                'mape': mean_absolute_percentage_error(y[:, i], y_pred[:, i]) * 100,
                'r2': r2_score(y[:, i], y_pred[:, i]),
                'max_error': np.max(np.abs(y[:, i] - y_pred[:, i])),
                'predictions': y_pred[:, i],
                'actuals': y[:, i]
            }
        
        return metrics

class XGBoostModel:
    """XGBoost model"""
    def __init__(self, **kwargs):
        default_params = {
            'n_estimators': 100,
            'learning_rate': 0.1,
            'max_depth': 6,
            'random_state': 42,
            'n_jobs': -1
        }
        default_params.update(kwargs)
        
        self.models = {
            'RSSI': xgb.XGBRegressor(**default_params),
            'SNR': xgb.XGBRegressor(**default_params),
            'path_loss': xgb.XGBRegressor(**default_params)
        }

    def train(self, X_train, y_train):
        """Train all models"""
        logger.info("Training XGBoost models...")
        for i, (name, model) in enumerate(self.models.items()):
            model.fit(X_train, y_train[:, i])
        logger.info("  XGBoost training completed!")

    def predict(self, X):
        """Make predictions"""
        predictions = np.zeros((X.shape[0], len(self.models)))
        for i, model in enumerate(self.models.values()):
            predictions[:, i] = model.predict(X)
        return predictions

    def evaluate(self, X, y):
        """Evaluate model and return detailed metrics"""
        from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
        
        y_pred = self.predict(X)
        metrics = {}
        
        for i, name in enumerate(['RSSI', 'SNR', 'path_loss']):
            metrics[name] = {
                'mse': mean_squared_error(y[:, i], y_pred[:, i]),
                'rmse': np.sqrt(mean_squared_error(y[:, i], y_pred[:, i])),
                'mae': mean_absolute_error(y[:, i], y_pred[:, i]),
                'mape': mean_absolute_percentage_error(y[:, i], y_pred[:, i]) * 100,
                'r2': r2_score(y[:, i], y_pred[:, i]),
                'max_error': np.max(np.abs(y[:, i] - y_pred[:, i])),
                'predictions': y_pred[:, i],
                'actuals': y[:, i]
            }
        
        return metrics

class EnsembleModel:
    """Ensemble combining multiple models"""
    def __init__(self, models_dict, device=None):
        self.models = models_dict
        self.device = device
        self.weights = None

    def calculate_optimal_weights(self, X_val, y_val):
        """Calculate optimal weights"""
        performances = {}
        for name, model in self.models.items():
            pred = self._predict_single(name, model, X_val)
            r2_scores = [r2_score(y_val[:, i], pred[:, i]) for i in range(y_val.shape[1])]
            avg_r2 = np.mean(r2_scores)
            performances[name] = max(0, avg_r2)  # Ensure non-negative
        
        total = sum(np.exp(r2 * 5) for r2 in performances.values())
        if total > 0:
            self.weights = {
                name: np.exp(performances[name] * 5) / total 
                for name in self.models.keys()
            }
        else:
            self.weights = {name: 1.0/len(self.models) for name in self.models.keys()}
        
        logger.info("Ensemble Weights:")
        for name, weight in self.weights.items():
            logger.info(f"  {name}: {weight:.3f}")

    def _predict_single(self, name, model, X):
        """Predict with single model"""
        if hasattr(model, 'eval'):  # Neural network
            model.eval()
            with torch.no_grad():
                X_tensor = torch.FloatTensor(X).to(self.device)
                return model(X_tensor).cpu().numpy()
        else:
            return model.predict(X)

    def predict(self, X):
        """Ensemble prediction"""
        if self.weights is None:
            self.weights = {name: 1.0/len(self.models) for name in self.models.keys()}
        
        predictions = {}
        for name, model in self.models.items():
            predictions[name] = self._predict_single(name, model, X)
        
        ensemble_pred = np.zeros_like(predictions[list(self.models.keys())[0]])
        for name, pred in predictions.items():
            ensemble_pred += pred * self.weights[name]
        
        return ensemble_pred

class BestModelSelector:
    """Evaluates and selects best model with COMPREHENSIVE METRICS"""
    def __init__(self, device):
        self.device = device
        self.models = {}
        self.performances = {}
        self.best_model = None
        self.best_name = None
        self.detailed_metrics = {}

    def add_model(self, name, model):
        """Add trained model"""
        self.models[name] = model

    def evaluate_all(self, X_test, y_test):
        """Evaluate all models with DETAILED METRICS"""
        from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, explained_variance_score
        
        logger.info("="*70)
        logger.info("COMPREHENSIVE MODEL EVALUATION")
        logger.info("="*70)
        
        metrics_names = ['RSSI', 'SNR', 'path_loss']
        
        for model_name, model in self.models.items():
            logger.info(f"\n{'='*70}")
            logger.info(f"MODEL: {model_name}")
            logger.info(f"{'='*70}")
            
            # Get predictions
            if hasattr(model, 'evaluate'):
                metrics = model.evaluate(X_test, y_test)
            else:
                # Fallback for models without evaluate method
                y_pred = model.predict(X_test)
                metrics = {}
                for i, metric_name in enumerate(metrics_names):
                    metrics[metric_name] = {
                        'mse': mean_squared_error(y_test[:, i], y_pred[:, i]),
                        'rmse': np.sqrt(mean_squared_error(y_test[:, i], y_pred[:, i])),
                        'mae': mean_absolute_error(y_test[:, i], y_pred[:, i]),
                        'mape': mean_absolute_percentage_error(y_test[:, i], y_pred[:, i]) * 100,
                        'r2': r2_score(y_test[:, i], y_pred[:, i]),
                        'max_error': np.max(np.abs(y_test[:, i] - y_pred[:, i])),
                        'explained_variance': explained_variance_score(y_test[:, i], y_pred[:, i])
                    }
            
            # Store detailed metrics
            self.detailed_metrics[model_name] = metrics
            
            # Display metrics for each target
            for metric_name in metrics_names:
                m = metrics[metric_name]
                logger.info(f"\n{metric_name} Prediction:")
                logger.info(f"  R² Score:           {m['r2']:.4f} (1.0 = perfect)")
                logger.info(f"  MSE:                {m['mse']:.4f}")
                logger.info(f"  RMSE:               {m['rmse']:.4f}")
                logger.info(f"  MAE:                {m['mae']:.4f}")
                logger.info(f"  MAPE:               {m['mape']:.2f}%")
                logger.info(f"  Max Error:          {m['max_error']:.4f}")
                if 'explained_variance' in m:
                    logger.info(f"  Explained Variance: {m['explained_variance']:.4f}")
            
            # Calculate aggregate performance
            avg_r2 = np.mean([metrics[m]['r2'] for m in metrics_names])
            avg_rmse = np.mean([metrics[m]['rmse'] for m in metrics_names])
            avg_mape = np.mean([metrics[m]['mape'] for m in metrics_names])
            
            performance = {
                'average_r2': avg_r2,
                'average_rmse': avg_rmse,
                'average_mape': avg_mape,
                'rssi_r2': metrics['RSSI']['r2'],
                'snr_r2': metrics['SNR']['r2'],
                'path_loss_r2': metrics['path_loss']['r2']
            }
            
            self.performances[model_name] = performance
            
            logger.info(f"\n{'='*70}")
            logger.info(f"OVERALL PERFORMANCE:")
            logger.info(f"  Average R²:    {avg_r2:.4f}")
            logger.info(f"  Average RMSE:  {avg_rmse:.4f}")
            logger.info(f"  Average MAPE:  {avg_mape:.2f}%")
            logger.info(f"{'='*70}")

    def print_comparison_table(self):
        """Print comparison table of all models"""
        logger.info("\n" + "="*70)
        logger.info("MODEL COMPARISON TABLE")
        logger.info("="*70)
        
        # Header
        header = f"{'Model':<20} {'Avg R²':<10} {'Avg RMSE':<10} {'Avg MAPE':<12} {'RSSI R²':<10} {'SNR R²':<10} {'PL R²':<10}"
        logger.info(header)
        logger.info("="*len(header))
        
        # Sort by average R²
        sorted_models = sorted(self.performances.items(), key=lambda x: x[1]['average_r2'], reverse=True)
        
        for model_name, perf in sorted_models:
            row = (
                f"{model_name:<20} "
                f"{perf['average_r2']:<10.4f} "
                f"{perf['average_rmse']:<10.4f} "
                f"{perf['average_mape']:<12.2f}% "
                f"{perf['rssi_r2']:<10.4f} "
                f"{perf['snr_r2']:<10.4f} "
                f"{perf['path_loss_r2']:<10.4f}")
            
            # Highlight best model
            if model_name == sorted_models[0][0]:
                logger.info(f" {row}")
            else:
                logger.info(f"  {row}")
        
        logger.info("="*70)

    def print_accuracy_interpretation(self):
        """Print interpretation of accuracy metrics"""
        logger.info("\n" + "="*70)
        logger.info("ACCURACY INTERPRETATION GUIDE")
        logger.info("="*70)
        
        logger.info("""
            R² Score (Coefficient of Determination):
            • 1.00      = Perfect predictions
            • 0.90-0.99 = Excellent
            • 0.80-0.89 = Very Good
            • 0.70-0.79 = Good
            • 0.60-0.69 = Moderate
            • < 0.60    = Needs Improvement

            RMSE (Root Mean Squared Error):
            • Lower is better
            • Same unit as target variable
            • Penalizes large errors more than MAE

            MAE (Mean Absolute Error):
            • Lower is better
            • Average prediction error
            • More robust to outliers than RMSE

            MAPE (Mean Absolute Percentage Error):
            • < 10%  = Highly accurate
            • 10-20% = Good
            • 20-50% = Reasonable
            • > 50%  = Poor
        """)
        logger.info("="*70)

    def create_ensemble(self, X_val, y_val):
        """Create ensemble model with FULL metrics"""
        logger.info("="*70)
        logger.info("CREATING ENSEMBLE MODEL")
        logger.info("="*70)
        if len(self.models) < 2:
            logger.warning("Need at least 2 models for ensemble")
            return

        ensemble = EnsembleModel(self.models, self.device)
        ensemble.calculate_optimal_weights(X_val, y_val)

        # Evaluate ensemble predictions
        y_pred = ensemble.predict(X_val)
        metrics_names = ['RSSI', 'SNR', 'path_loss']
        metrics = {}

        from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
        import numpy as np

        for i, name in enumerate(metrics_names):
            mse = mean_squared_error(y_val[:, i], y_pred[:, i])
            rmse = np.sqrt(mse)
            mape = mean_absolute_percentage_error(y_val[:, i], y_pred[:, i]) * 100
            r2 = r2_score(y_val[:, i], y_pred[:, i])
            metrics[name] = {
                'mse': mse,
                'rmse': rmse,
                'mape': mape,
                'r2': r2
            }

        # Compute aggregate metrics
        avg_r2 = np.mean([metrics[m]['r2'] for m in metrics_names])
        avg_rmse = np.mean([metrics[m]['rmse'] for m in metrics_names])
        avg_mape = np.mean([metrics[m]['mape'] for m in metrics_names])

        # Store FULL performance dict (matching other models)
        self.performances['Ensemble'] = {
            'average_r2': avg_r2,
            'average_rmse': avg_rmse,
            'average_mape': avg_mape,
            'rssi_r2': metrics['RSSI']['r2'],
            'snr_r2': metrics['SNR']['r2'],
            'path_loss_r2': metrics['path_loss']['r2']
        }

        logger.info("Ensemble Performance:")
        logger.info(f"  Average R²:   {avg_r2:.4f}")
        logger.info(f"  Average RMSE: {avg_rmse:.4f}")
        logger.info(f"  Average MAPE: {avg_mape:.2f}%")

        self.models['Ensemble'] = ensemble

    def select_best(self):
        """Select best model"""
        logger.info("="*70)
        logger.info("SELECTING BEST MODEL")
        logger.info("="*70)
        
        best_r2 = -np.inf
        for name, perf in self.performances.items():
            if perf['average_r2'] > best_r2:
                best_r2 = perf['average_r2']
                self.best_name = name
                self.best_model = self.models[name]
        
        logger.info(f"BEST MODEL: {self.best_name}")
        logger.info(f"Average R²: {best_r2:.4f}")
        
        return self.best_model, self.best_name

    def save_best_model(self, scaler, feature_cols, output_dir='./models'):
        """Save best model"""
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True, parents=True)
        
        model_file = output_path / f'{self.best_name.lower()}_best_model.pkl'
        with open(model_file, 'wb') as f:
            pickle.dump(self.best_model, f)
        logger.info(f"Saved: {model_file}")
        
        scaler_file = output_path / 'scaler.pkl'
        with open(scaler_file, 'wb') as f:
            pickle.dump(scaler, f)
        
        metadata = {
            'best_model_name': self.best_name,
            'performance': self.performances[self.best_name],
            'all_performances': self.performances,
            'feature_columns': feature_cols
        }
        
        metadata_file = output_path / 'model_metadata.json'
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)
        logger.info(f"Saved: {metadata_file}")

    def get_feature_importance(self, feature_names):
        """Extract feature importance from best model"""
        logger.info("Extracting feature importance...")
        
        if 'Random_Forest' in self.models:
            rf_model = self.models['Random_Forest']
            # Average importance across all 3 models (RSSI, SNR, path_loss)
            importances = np.mean([
                rf_model.models['RSSI'].feature_importances_,
                rf_model.models['SNR'].feature_importances_,
                rf_model.models['path_loss'].feature_importances_
            ], axis=0)
            
            importance_data = {
                'feature': feature_names,
                'importance': importances
            }
            return importance_data
        
        elif 'XGBoost' in self.models:
            xgb_model = self.models['XGBoost']
            # Average importance across all 3 models
            importances = np.mean([
                xgb_model.models['RSSI'].feature_importances_,
                xgb_model.models['SNR'].feature_importances_,
                xgb_model.models['path_loss'].feature_importances_
            ], axis=0)
            
            importance_data = {
                'feature': feature_names,
                'importance': importances
            }
            return importance_data
        
        else:
            logger.warning("Feature importance not available for Neural Network")
            return None


### Path Optimization using A*

In [14]:
class PathOptimizer:
    """
    FIXED: Better cost calculation and realistic predictions
    """
    
    def __init__(self, model, scaler, feature_cols, gee_integration, config):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.gee = gee_integration
        self.config = config
        self.physics_engine = LoRaPhysicsEngine()
        self.feature_builder = UnifiedFeatureBuilder()

    def calculate_distance(self, lat1, lon1, lat2, lon2):
        """Calculate Haversine distance in meters"""
        R = 6371000
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dphi = np.radians(lat2 - lat1)
        dlambda = np.radians(lon2 - lon1)
        a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        return R * c

    def _calculate_bearing(self, lat1, lon1, lat2, lon2):
        """Calculate bearing between two points"""
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlon = lon2 - lon1
        x = np.sin(dlon) * np.cos(lat2)
        y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
        bearing = np.arctan2(x, y)
        return (np.degrees(bearing) + 360) % 360

    def _destination_point(self, lat, lon, distance_m, bearing_deg):
        """Calculate destination point given distance and bearing"""
        R = 6371000
        lat1 = np.radians(lat)
        lon1 = np.radians(lon)
        brng = np.radians(bearing_deg)
        d = distance_m / R
        
        lat2 = np.arcsin(np.sin(lat1) * np.cos(d) + np.cos(lat1) * np.sin(d) * np.cos(brng))
        lon2 = lon1 + np.arctan2(
            np.sin(brng) * np.sin(d) * np.cos(lat1),
            np.cos(d) - np.sin(lat1) * np.sin(lat2)
        )
        
        return np.degrees(lat2), np.degrees(lon2)
    
    def generate_adaptive_grid(self, start_lat, start_lon, dest_lat, dest_lon):
        """Generate adaptive grid"""
        total_distance = self.calculate_distance(start_lat, start_lon, dest_lat, dest_lon)
        bearing = self._calculate_bearing(start_lat, start_lon, dest_lat, dest_lon)
        perpendicular_bearing = (bearing + 90) % 360
        
        segment_spacing_m = self.config.grid_spacing_km * 1000
        num_segments = max(1, int(np.floor(total_distance / segment_spacing_m)) - 1)
        
        if self.config.adaptive_grid:
            if total_distance < 3000:
                num_lanes = 9
            elif total_distance < 8000:
                num_lanes = 11
            else:
                num_lanes = 15
            corridor_width_km = self.config.corridor_width_km
        else:
            corridor_width_km = self.config.corridor_width_km
            num_lanes = 11
        
        logger.info(f"  Grid Configuration:")
        logger.info(f"    Distance: {total_distance/1000:.2f} km")
        logger.info(f"    Segments: {num_segments}")
        logger.info(f"    Corridor width: ±{corridor_width_km/2:.2f} km")
        logger.info(f"    Lanes: {num_lanes}")
        logger.info(f"    Total points: {num_segments * num_lanes}")
        
        grid_points = []
        coordinates = []
        lane_offsets = np.linspace(-corridor_width_km/2, corridor_width_km/2, num_lanes) * 1000
        
        # Generate grid points BETWEEN TX and RX
        for segment_idx in range(num_segments):
            # Start from segment 1 and end at segment (total-1)
            progress = (segment_idx + 1) / (num_segments + 1)
            center_lat = start_lat + progress * (dest_lat - start_lat)
            center_lon = start_lon + progress * (dest_lon - start_lon)
            
            for lane_idx, offset_m in enumerate(lane_offsets):
                lat, lon = self._destination_point(center_lat, center_lon, offset_m, perpendicular_bearing)
                
                point = PathPoint(
                    lat=lat,
                    lon=lon,
                    grid_x=segment_idx,
                    grid_y=lane_idx
                )
                
                grid_points.append(point)
                coordinates.append((lat, lon))
        
        return grid_points, coordinates, num_segments, num_lanes
    
    def _batch_predict_all_hops(self, grid_points, num_segments, num_lanes, 
                                lora_params, start_lat, start_lon):
        """OPTIMIZED: Batch predict all hops with parallel path feature fetching"""
        logger.info("Pre-computing ALL hop predictions...")
        
        # ============================================================
        # STEP 1: COLLECT ALL PATH PAIRS FIRST
        # ============================================================
        all_features = []
        hop_map = {}
        path_pairs = []
        hop_to_path_idx = {}
        
        total_hops = 0
        
        # TX → first segment points (grid_x=0)
        for lane_idx in range(num_lanes):
            first_seg_idx = lane_idx  # grid_x=0, grid_y=lane_idx
            first_point = grid_points[first_seg_idx]
            
            # Store path pair for batch fetching
            path_pair = ((start_lat, start_lon), (first_point.lat, first_point.lon))
            hop_to_path_idx[(-1, first_seg_idx)] = len(path_pairs)
            path_pairs.append(path_pair)
            total_hops += 1
        
        # Grid→grid path pairs (existing code)
        for seg_idx in range(num_segments - 1):
            for curr_lane in range(num_lanes):
                curr_idx = seg_idx * num_lanes + curr_lane
                curr_point = grid_points[curr_idx]
                
                for next_lane in range(max(0, curr_lane - 3), min(num_lanes, curr_lane + 4)):
                    next_idx = (seg_idx + 1) * num_lanes + next_lane
                    next_point = grid_points[next_idx]
                    
                    path_pair = ((curr_point.lat, curr_point.lon), (next_point.lat, next_point.lon))
                    hop_to_path_idx[(curr_idx, next_idx)] = len(path_pairs)
                    path_pairs.append(path_pair)
                    total_hops += 1
                    
        logger.info(f"  Total hops to predict: {total_hops}")
        
        # ============================================================
        # STEP 2: BATCH FETCH ALL PATH FEATURES (PARALLEL + CACHED)
        # ============================================================
        path_features_list = self.gee.batch_get_path_spatial_features(path_pairs)
        
        logger.info(f"  Unique path pairs: {len(set(path_pairs))}")
        logger.info(f"  Cache hit rate will be ~{(1 - len(set(path_pairs))/len(path_pairs))*100:.1f}%")
        # ============================================================
        # STEP 3: BUILD FEATURE VECTORS
        # ============================================================
        logger.info(f"  Building feature vectors...")
        
        for lane_idx in range(num_lanes):
            first_seg_idx = lane_idx
            first_point = grid_points[first_seg_idx]
            
            dist = self.calculate_distance(start_lat, start_lon, first_point.lat, first_point.lon)
            
            # Get pre-fetched path features
            path_idx = hop_to_path_idx[(-1, first_seg_idx)]
            path_feats = path_features_list[path_idx]
            
            # Build feature vector
            features = np.array([
                first_point.elevation,
                first_point.land_cover,
                first_point.terrain_penalty,
                dist,
                lora_params.spreading_factor,
                lora_params.frequency,
                lora_params.tx_power,
                first_point.elevation / 1000.0,
                path_feats['path_built_up_fraction'],
                path_feats['path_vegetation_fraction'],
                path_feats['path_water_fraction'],
                path_feats['path_avg_penalty'],
                path_feats['path_elevation_std'],
                path_feats['max_terrain_obstruction_m'],
                path_feats['path_dominant_land_cover']
            ])
            
            hop_map[(-1, first_seg_idx)] = len(all_features)
            all_features.append(features)
        
        for seg_idx in range(num_segments - 1):
            for curr_lane in range(num_lanes):
                curr_idx = seg_idx * num_lanes + curr_lane
                curr_point = grid_points[curr_idx]
                
                for next_lane in range(max(0, curr_lane - 3), min(num_lanes, curr_lane + 4)):
                    next_idx = (seg_idx + 1) * num_lanes + next_lane
                    next_point = grid_points[next_idx]
                    
                    dist = self.calculate_distance(
                        curr_point.lat, curr_point.lon,
                        next_point.lat, next_point.lon
                    )
                    
                    # Get pre-fetched path features
                    path_idx = hop_to_path_idx[(curr_idx, next_idx)]
                    path_feats = path_features_list[path_idx]
                    
                    # Build feature vector WITH REAL PATH FEATURES
                    features = np.array([
                        next_point.elevation,
                        next_point.land_cover,
                        next_point.terrain_penalty,
                        dist,
                        lora_params.spreading_factor,
                        lora_params.frequency,
                        lora_params.tx_power,
                        next_point.elevation / 1000.0,
                        path_feats['path_built_up_fraction'],
                        path_feats['path_vegetation_fraction'],
                        path_feats['path_water_fraction'],
                        path_feats['path_avg_penalty'],
                        path_feats['path_elevation_std'],
                        path_feats['max_terrain_obstruction_m'],
                        path_feats['path_dominant_land_cover']
                    ])
                    
                    hop_map[(curr_idx, next_idx)] = len(all_features)
                    all_features.append(features)
        
        # ============================================================
        # STEP 4: BATCH PREDICT WITH ML MODEL
        # ============================================================
        hop_predictions = {}
        if all_features:
            X = np.array(all_features)
            X_scaled = self.scaler.transform(X)
            
            logger.info(f"  Running batch ML prediction...")
            predictions = self.model.predict(X_scaled)
            logger.info(f"  Predictions complete!")
            
            # Store predictions in hop_predictions dict
            for (curr_idx, next_idx), pred_idx in hop_map.items():
                rssi = np.clip(predictions[pred_idx][0], -150, -20)
                snr = predictions[pred_idx][1]
                path_loss = predictions[pred_idx][2]
                
                next_point = grid_points[next_idx]
                
                # Calculate PDR from SNR
                pdr = self.physics_engine.calculate_pdr(
                    snr,
                    lora_params.spreading_factor,
                    next_point.land_cover
                )
                
                hop_predictions[(curr_idx, next_idx)] = {
                    'rssi': rssi,
                    'snr': snr,
                    'path_loss': path_loss,
                    'pdr': pdr
                }
        
        # In _batch_predict_all_hops(), after storing predictions:
        for lane_idx in range(num_lanes):
            first_seg_idx = lane_idx
            if (-1, first_seg_idx) in hop_predictions:
                pred = hop_predictions[(-1, first_seg_idx)]
                grid_points[first_seg_idx].pdr = pred['pdr']
                grid_points[first_seg_idx].rssi = pred['rssi']
                grid_points[first_seg_idx].snr = pred['snr']
                grid_points[first_seg_idx].path_loss = pred['path_loss']
                
        # ============================================================
        # STEP 5: UPDATE GRID_POINTS WITH PREDICTIONS
        # ============================================================
        logger.info(f"  Updating grid points with predictions...")
        
        for idx in range(len(grid_points)):
            best_pdr = 0.0
            best_rssi = -120.0
            best_snr = -10.0
            best_path_loss = 120.0
            
            for (src_idx, dst_idx), pred in hop_predictions.items():
                if dst_idx == idx:
                    if pred['pdr'] > best_pdr:
                        best_pdr = pred['pdr']
                        best_rssi = pred['rssi']
                        best_snr = pred['snr']
                        best_path_loss = pred['path_loss']
            
            if best_pdr > 0:
                grid_points[idx].pdr = best_pdr
                grid_points[idx].rssi = best_rssi
                grid_points[idx].snr = best_snr
                grid_points[idx].path_loss = best_path_loss
        
        logger.info(f"  All {total_hops} hop predictions stored!")
        logger.info(f"  Grid points updated with predictions!")
        
        return hop_predictions

    def calculate_lora_cost(self, pdr, terrain_penalty, distance, land_cover):
        """
        FIXED: Better cost function that properly penalizes buildings
        """
        # CRITICAL FIX: Heavy penalty for low PDR
        if pdr < self.config.min_pdr_threshold:
            return 10000.0  # Blocked
        elif pdr < 0.4:
            pdr_cost = 100.0
        elif pdr < 0.6:
            pdr_cost = 20.0
        elif pdr < 0.8:
            pdr_cost = 5.0
        else:
            pdr_cost = 0.5
        
        # Distance cost (normalized)
        distance_cost = distance / 1000.0
        
        # FIXED: Stronger terrain penalty
        terrain_cost = terrain_penalty * 10.0
        
        # Apply preferences
        if self.config.prefer_water and land_cover == 80:
            terrain_cost *= 0.1
        if self.config.avoid_buildings and land_cover == 50:
            terrain_cost *= 5.0  # FIXED: Much stronger penalty for buildings
        
        total_cost = pdr_cost * 0.7 + distance_cost * 0.1 + terrain_cost * 0.2
        
        return total_cost
    
    def find_optimal_path(self, start_lat, start_lon, dest_lat, dest_lon,
                        lora_params, config=None):
        """
        FIXED A* pathfinding
        """
        if config:
            self.config = config
        
        logger.info("="*70)
        logger.info("PATH OPTIMIZATION WITH A*")
        logger.info("="*70)
        
        # Step 1: Generate grid
        logger.info("[1/4] Generating grid...")
        grid_points, coordinates, num_segments, num_lanes = \
            self.generate_adaptive_grid(start_lat, start_lon, dest_lat, dest_lon)
        
        # Step 2: Fetch spatial data
        logger.info("[2/4] Fetching spatial data from GEE...")
        spatial_results = self.gee.batch_fetch_spatial_features(coordinates)
        
        for i, spatial in enumerate(spatial_results):
            grid_points[i].elevation = spatial['elevation']
            grid_points[i].land_cover = spatial['land_cover']
            grid_points[i].terrain_penalty = spatial['terrain_penalty']
        
        # Step 3: Pre-compute predictions
        logger.info("[3/4] Pre-computing predictions...")
        hop_predictions = self._batch_predict_all_hops(
            grid_points, num_segments, num_lanes, lora_params,
            start_lat, start_lon 
        )
        
        # Step 4: A* pathfinding
        logger.info("[4/4] Running A* pathfinding...")
        
        # Find start node
        start_candidates = [p for p in grid_points if p.grid_x == 0]
        start_node = min(start_candidates, key=lambda p: abs(p.grid_y - num_lanes//2))
        start_idx = start_node.grid_x * num_lanes + start_node.grid_y
        
        # Calculate heuristics
        for point in grid_points:
            point.distance_to_goal = self.calculate_distance(
                point.lat, point.lon, dest_lat, dest_lon
            )
        
        open_set = []
        heapq.heappush(open_set, (0, start_idx))
        closed_set = set()
        came_from = {}
        g_score = {start_idx: 0}
        f_score = {start_idx: start_node.distance_to_goal / 10000}
        
        iterations = 0
        
        while open_set:
            iterations += 1
            
            if iterations % 50 == 0:
                _, current_idx = open_set[0]
                current_seg = (current_idx // num_lanes)
                logger.info(f"  Progress: Segment {current_seg}/{num_segments-1}, Iteration {iterations}")
            
            _, current_idx = heapq.heappop(open_set)
            
            if current_idx in closed_set:
                continue
            
            current_seg = current_idx // num_lanes
            
            # Reached destination?
            if current_seg == num_segments - 1:
                logger.info(f"  PATH FOUND!")
                
                # Reconstruct path
                path_indices = [current_idx]
                while current_idx in came_from:
                    current_idx = came_from[current_idx]
                    path_indices.insert(0, current_idx)
                
                # Path indices only contain beacons between TX and RX
                path = [grid_points[idx] for idx in path_indices]
                
                # Apply predictions to path points
                for i in range(len(path)):
                    if i > 0:
                        prev_idx = path_indices[i-1]
                        curr_idx = path_indices[i]
                        if (prev_idx, curr_idx) in hop_predictions:
                            pred = hop_predictions[(prev_idx, curr_idx)]
                            path[i].rssi = pred['rssi']
                            path[i].snr = pred['snr']
                            path[i].path_loss = pred['path_loss']
                            path[i].pdr = pred['pdr']
                
                # Statistics
                all_pdrs = [p.pdr for p in path if p.pdr > 0]
                all_rssi = [p.rssi for p in path]
                all_snr = [p.snr for p in path]

                if path:
                    last_beacon = path[-1]
                    final_hop = self.predict_hop(last_beacon.lat, last_beacon.lon, dest_lat, dest_lon, lora_params)
                    all_pdrs.append(final_hop.pdr)
                    all_rssi.append(final_hop.rssi)
                    all_snr.append(final_hop.snr)
                else:  # checks if path is empty
                    final_hop = self.predict_hop(start_lat, start_lon, dest_lat, dest_lon, lora_params)
                    all_pdrs = [final_hop.pdr]
                    all_rssi = [final_hop.rssi]
                    all_snr = [final_hop.snr]

                avg_pdr = np.mean(all_pdrs)
                min_pdr = min(all_pdrs)
                avg_snr = np.mean(all_snr)
                avg_rssi = np.mean(all_rssi)
                
                # rx point with full features
                rx_point = final_hop
                
                logger.info(f"  Iterations: {iterations}")
                logger.info(f"  Beacons: {len(path)}")
                logger.info(f"  Avg PDR: {avg_pdr:.3f} ({avg_pdr*100:.1f}%)")
                logger.info(f"  Min PDR: {min_pdr:.3f} ({min_pdr*100:.1f}%)")
                logger.info(f"  Avg SNR: {avg_snr:.2f} dB")
                logger.info(f"  Avg RSSI: {avg_rssi:.1f} dBm")
                
                return path, grid_points, rx_point
            
            closed_set.add(current_idx)
            
            # Explore neighbors
            current_lane = current_idx % num_lanes
            neighbor_lanes = range(
                max(0, current_lane - 3),
                min(num_lanes, current_lane + 4)
            )
            
            for lane in neighbor_lanes:
                neighbor_idx = (current_seg + 1) * num_lanes + lane
                
                if neighbor_idx >= len(grid_points) or neighbor_idx in closed_set:
                    continue
                
                # Get prediction
                if (current_idx, neighbor_idx) not in hop_predictions:
                    continue
                
                pred = hop_predictions[(current_idx, neighbor_idx)]
                neighbor = grid_points[neighbor_idx]
                
                distance = self.calculate_distance(
                    grid_points[current_idx].lat, grid_points[current_idx].lon,
                    neighbor.lat, neighbor.lon
                )
                
                cost = self.calculate_lora_cost(
                    pred['pdr'], 
                    neighbor.terrain_penalty, 
                    distance,
                    neighbor.land_cover
                )
                
                # Penalize zigzagging
                lane_diff = abs(lane - current_lane)
                if lane_diff > 2:
                    cost += 0.5 * lane_diff
                
                tentative_g = g_score[current_idx] + cost
                
                if neighbor_idx not in g_score or tentative_g < g_score[neighbor_idx]:
                    came_from[neighbor_idx] = current_idx
                    g_score[neighbor_idx] = tentative_g
                    f = tentative_g + neighbor.distance_to_goal / 10000
                    f_score[neighbor_idx] = f
                    heapq.heappush(open_set, (f, neighbor_idx))
        
        raise RuntimeError(
            f"No viable path found after {iterations} iterations. "
            f"Try: increasing corridor_width_km or lowering min_pdr_threshold"
        )
    
    def predict_hop(self, tx_lat, tx_lon, rx_lat, rx_lon, lora_params):
        """Predict link quality for a SINGLE HOP"""
        hop_distance = self.calculate_distance(tx_lat, tx_lon, rx_lat, rx_lon)
        tx_features = self.gee.get_spatial_features(tx_lat, tx_lon)
        path_feats = self.gee.get_path_spatial_features(tx_lat, tx_lon, rx_lat, rx_lon)
        
        rx_point = PathPoint(
            lat=rx_lat, lon=rx_lon,
            elevation=tx_features['elevation'],
            land_cover=tx_features['land_cover'],
            terrain_penalty=tx_features['terrain_penalty'],
            distance_to_start=hop_distance,
            path_built_up_fraction=path_feats['path_built_up_fraction'],
            path_vegetation_fraction=path_feats['path_vegetation_fraction'],
            path_water_fraction=path_feats['path_water_fraction'],
            path_avg_penalty=path_feats['path_avg_penalty'],
            path_elevation_std=path_feats['path_elevation_std'],
            max_terrain_obstruction_m=path_feats['max_terrain_obstruction_m'],
            path_dominant_land_cover=path_feats['path_dominant_land_cover']
        )
        
        features = self.feature_builder.build_feature_vector(rx_point, lora_params)
        features_scaled = self.scaler.transform(features)
        predictions = self.model.predict(features_scaled)[0]
        
        rx_point.rssi = np.clip(predictions[0], -150, -20)
        rx_point.snr = predictions[1]
        rx_point.path_loss = predictions[2]
        rx_point.pdr = self.physics_engine.calculate_pdr(
            rx_point.snr, lora_params.spreading_factor, rx_point.land_cover
        )
        
        return rx_point
    
    def sample_direct_path(self, start_lat, start_lon, dest_lat, dest_lon, 
                      lora_params, num_samples=None, beacon_spacing_km=None,
                      use_grid_alignment=True):
        """
        Sample points along direct path for comparison
        
        Args:
            start_lat, start_lon: Starting coordinates
            dest_lat, dest_lon: Destination coordinates
            lora_params: LoRa parameters
            num_samples: Fixed number of samples (overrides other options)
            beacon_spacing_km: Distance between beacons in km (default: match grid_spacing_km)
            use_grid_alignment: If True, align with grid center lane; if False, use spacing/samples
        
        Returns:
            Dictionary with direct path metrics and points
        """
        total_distance = self.calculate_distance(start_lat, start_lon, dest_lat, dest_lon)
        
        # ========================================================================
        # OPTION 1: GRID ALIGNMENT (Perfect match with optimization grid)
        # ========================================================================
        if use_grid_alignment:
            logger.info(f"Sampling direct path (GRID-ALIGNED with center lane)...")
            
            # Generate the same grid structure as optimization
            grid_points, coordinates, num_segments, num_lanes = \
                self.generate_adaptive_grid(start_lat, start_lon, dest_lat, dest_lon)
            
            # Fetch spatial data for grid
            logger.info(f"  Fetching spatial data for {len(coordinates)} grid points...")
            spatial_results = self.gee.batch_fetch_spatial_features(coordinates)
            
            for i, spatial in enumerate(spatial_results):
                grid_points[i].elevation = spatial['elevation']
                grid_points[i].land_cover = spatial['land_cover']
                grid_points[i].terrain_penalty = spatial['terrain_penalty']
            
            # Extract CENTER LANE points (middle of corridor)
            center_lane_idx = num_lanes // 2
            direct_points = []
            
            logger.info(f"  Extracting center lane (lane {center_lane_idx}/{num_lanes-1})...")
            
            for seg_idx in range(num_segments):
                point_idx = seg_idx * num_lanes + center_lane_idx
                point = grid_points[point_idx]
                
                # Get path features from start to this point
                if seg_idx > 0:
                    path_feats = self.gee.get_path_spatial_features(
                        start_lat, start_lon, point.lat, point.lon
                    )
                else:
                    path_feats = {
                        'path_built_up_fraction': 0.0,
                        'path_vegetation_fraction': 0.0,
                        'path_water_fraction': 0.0,
                        'path_avg_penalty': 0.3,
                        'path_elevation_std': 0.0,
                        'max_terrain_obstruction_m': 0.0,
                        'path_dominant_land_cover': 50
                    }
                
                # Update point with path features
                point.path_built_up_fraction = path_feats['path_built_up_fraction']
                point.path_vegetation_fraction = path_feats['path_vegetation_fraction']
                point.path_water_fraction = path_feats['path_water_fraction']
                point.path_avg_penalty = path_feats['path_avg_penalty']
                point.path_elevation_std = path_feats['path_elevation_std']
                point.max_terrain_obstruction_m = path_feats['max_terrain_obstruction_m']
                point.path_dominant_land_cover = path_feats['path_dominant_land_cover']
                point.distance_to_start = self.calculate_distance(
                    start_lat, start_lon, point.lat, point.lon
                )
                
                # Predict link quality
                features = self.feature_builder.build_feature_vector(point, lora_params)
                features_scaled = self.scaler.transform(features)
                predictions = self.model.predict(features_scaled)[0]
                
                point.rssi = np.clip(predictions[0], -150, -20)
                point.snr = predictions[1]
                point.path_loss = predictions[2]
                point.pdr = self.physics_engine.calculate_pdr(
                    point.snr, lora_params.spreading_factor, point.land_cover
                )
                
                direct_points.append(point)
            
            logger.info(f"  Sampled {len(direct_points)} points (grid spacing: {self.config.grid_spacing_km} km)")
        
        # ========================================================================
        # OPTION 2: CUSTOM SPACING/SAMPLES (Independent from grid)
        # ========================================================================
        else:
            # Determine sampling strategy
            if num_samples is not None:
                # Fixed number of samples
                sample_count = num_samples
                logger.info(f"Sampling direct path ({sample_count} FIXED samples)...")
            elif beacon_spacing_km is not None:
                # Based on beacon spacing
                sample_count = max(3, int(np.ceil(total_distance / (beacon_spacing_km * 1000))))
                logger.info(f"Sampling direct path ({beacon_spacing_km} km spacing = {sample_count} points)...")
            else:
                # Default: match grid spacing
                beacon_spacing_km = self.config.grid_spacing_km
                sample_count = max(3, int(np.ceil(total_distance / (beacon_spacing_km * 1000))))
                logger.info(f"Sampling direct path (DEFAULT spacing: {beacon_spacing_km} km = {sample_count} points)...")
            
            # Generate sample points
            lats = np.linspace(start_lat, dest_lat, sample_count)
            lons = np.linspace(start_lon, dest_lon, sample_count)
            direct_points = []
            
            for i, (lat, lon) in enumerate(zip(lats, lons)):
                try:
                    spatial = self.gee.get_spatial_features(lat, lon)
                    
                    if i > 0:
                        path_feats = self.gee.get_path_spatial_features(start_lat, start_lon, lat, lon)
                    else:
                        path_feats = {
                            'path_built_up_fraction': 0.0,
                            'path_vegetation_fraction': 0.0,
                            'path_water_fraction': 0.0,
                            'path_avg_penalty': 0.3,
                            'path_elevation_std': 0.0,
                            'max_terrain_obstruction_m': 0.0,
                            'path_dominant_land_cover': 50
                        }
                    
                    point = PathPoint(
                        lat=lat, lon=lon,
                        elevation=spatial['elevation'],
                        land_cover=spatial['land_cover'],
                        terrain_penalty=spatial['terrain_penalty'],
                        distance_to_start=self.calculate_distance(start_lat, start_lon, lat, lon),
                        path_built_up_fraction=path_feats['path_built_up_fraction'],
                        path_vegetation_fraction=path_feats['path_vegetation_fraction'],
                        path_water_fraction=path_feats['path_water_fraction'],
                        path_avg_penalty=path_feats['path_avg_penalty'],
                        path_elevation_std=path_feats['path_elevation_std'],
                        max_terrain_obstruction_m=path_feats['max_terrain_obstruction_m'],
                        path_dominant_land_cover=path_feats['path_dominant_land_cover']
                    )
                    
                    features = self.feature_builder.build_feature_vector(point, lora_params)
                    features_scaled = self.scaler.transform(features)
                    predictions = self.model.predict(features_scaled)[0]
                    
                    point.rssi = np.clip(predictions[0], -150, -20)
                    point.snr = predictions[1]
                    point.path_loss = predictions[2]
                    point.pdr = self.physics_engine.calculate_pdr(
                        point.snr, lora_params.spreading_factor, point.land_cover
                    )
                    
                    direct_points.append(point)
                    
                except Exception as e:
                    logger.warning(f"Failed at point {i}: {e}")
                    continue
            
            if not direct_points:
                raise RuntimeError("Failed to sample direct path")
            
            # Log actual spacing
            if len(direct_points) > 1:
                actual_spacing = (total_distance / 1000) / (len(direct_points) - 1)
                logger.info(f"  Actual spacing: {actual_spacing:.2f} km between {len(direct_points)} points")
        
        # ========================================================================
        # CALCULATE STATISTICS
        # ========================================================================
        all_pdrs = [p.pdr for p in direct_points]
        all_rssi = [p.rssi for p in direct_points]
        all_snr = [p.snr for p in direct_points]
        all_path_loss = [p.path_loss for p in direct_points]
        
        # Predict final hop
        if direct_points:
            last = direct_points[-1]
            final_hop = self.predict_hop(last.lat, last.lon, dest_lat, dest_lon, lora_params)
            all_pdrs.append(final_hop.pdr)
            all_rssi.append(final_hop.rssi)
            all_snr.append(final_hop.snr)
            all_path_loss.append(final_hop.path_loss)
        else:
            # Very short path: direct TX→RX
            direct_link = self.predict_hop(start_lat, start_lon, dest_lat, dest_lon, lora_params)
            all_pdrs = [direct_link.pdr]
            all_rssi = [direct_link.rssi]
            all_snr = [direct_link.snr]
            all_path_loss = [direct_link.path_loss]

        avg_pdr = np.mean(all_pdrs)
        avg_rssi = np.mean(all_rssi)
        avg_snr = np.mean(all_snr)
        avg_path_loss = np.mean(all_path_loss)
        
        # rx_point with full features
        rx_point = final_hop

        logger.info(f"  Direct path: PDR={avg_pdr:.3f}, RSSI={avg_rssi:.1f}dBm, SNR={avg_snr:.2f}dB")
        return {
            'RSSI': avg_rssi,
            'SNR': avg_snr,
            'PDR': avg_pdr,
            'path_loss': avg_path_loss,
            'points': direct_points,
            'rx_point': rx_point
        }


### Visualization

In [15]:
class ResultVisualizer:
    """Visualization tools for path optimization results"""
    
    def __init__(self):
        plt.style.use('seaborn-v0_8-darkgrid')
        self.output_dir = Path("./output")
        self.output_dir.mkdir(exist_ok=True)
    
    def export_results_to_csv(self, optimal_path, grid_points, direct_path_points,
                          model_performances, optimal_rx_point, direct_rx_point,
                          feature_importance_data=None):
        """Export all results to CSV files, including RX with full metrics"""
        logger.info("Exporting results to CSV...")

        # 1. Export Optimal Path (including RX)
        optimal_path_data = []
        for i, point in enumerate(optimal_path):
            optimal_path_data.append({
                'beacon_number': i + 1,
                'latitude': point.lat,
                'longitude': point.lon,
                'elevation_m': point.elevation,
                'land_cover': point.land_cover,
                'terrain_penalty': point.terrain_penalty,
                'rssi_dbm': point.rssi,
                'snr_db': point.snr,
                'pdr': point.pdr,
                'path_loss_db': point.path_loss,
                'path_built_up_fraction': point.path_built_up_fraction,
                'path_vegetation_fraction': point.path_vegetation_fraction,
                'path_water_fraction': point.path_water_fraction,
                'path_avg_penalty': point.path_avg_penalty,
                'path_elevation_std': point.path_elevation_std,
                'max_terrain_obstruction_m': point.max_terrain_obstruction_m
            })
        # Append RX with REAL predicted values
        optimal_path_data.append({
            'beacon_number': 'RX',
            'latitude': optimal_rx_point.lat,
            'longitude': optimal_rx_point.lon,
            'elevation_m': optimal_rx_point.elevation,
            'land_cover': optimal_rx_point.land_cover,
            'terrain_penalty': optimal_rx_point.terrain_penalty,
            'rssi_dbm': optimal_rx_point.rssi,
            'snr_db': optimal_rx_point.snr,
            'pdr': optimal_rx_point.pdr,
            'path_loss_db': optimal_rx_point.path_loss,
            'path_built_up_fraction': optimal_rx_point.path_built_up_fraction,
            'path_vegetation_fraction': optimal_rx_point.path_vegetation_fraction,
            'path_water_fraction': optimal_rx_point.path_water_fraction,
            'path_avg_penalty': optimal_rx_point.path_avg_penalty,
            'path_elevation_std': optimal_rx_point.path_elevation_std,
            'max_terrain_obstruction_m': optimal_rx_point.max_terrain_obstruction_m
        })
        df_optimal = pd.DataFrame(optimal_path_data)
        optimal_file = self.output_dir / 'optimal_path.csv'
        df_optimal.to_csv(optimal_file, index=False)
        logger.info(f"  Optimal path saved: {optimal_file}")

        # 2. Export Grid Points
        grid_data = []
        for point in grid_points:
            grid_data.append({
                'grid_x': point.grid_x,
                'grid_y': point.grid_y,
                'latitude': point.lat,
                'longitude': point.lon,
                'elevation_m': point.elevation,
                'land_cover': point.land_cover,
                'terrain_penalty': point.terrain_penalty,
                'rssi_dbm': point.rssi,
                'snr_db': point.snr,
                'pdr': point.pdr,
                'path_loss_db': point.path_loss
            })
        df_grid = pd.DataFrame(grid_data)
        grid_file = self.output_dir / 'grid_points.csv'
        df_grid.to_csv(grid_file, index=False)
        logger.info(f"  Grid points saved: {grid_file}")

        # 3. Export Direct Path Points (including RX)
        direct_data = []
        for i, point in enumerate(direct_path_points):
            direct_data.append({
                'sample_number': i + 1,
                'latitude': point.lat,
                'longitude': point.lon,
                'elevation_m': point.elevation,
                'land_cover': point.land_cover,
                'terrain_penalty': point.terrain_penalty,
                'rssi_dbm': point.rssi,
                'snr_db': point.snr,
                'pdr': point.pdr,
                'path_loss_db': point.path_loss
            })
        # Append RX with REAL predicted values
        direct_data.append({
            'sample_number': 'RX',
            'latitude': direct_rx_point.lat,
            'longitude': direct_rx_point.lon,
            'elevation_m': direct_rx_point.elevation,
            'land_cover': direct_rx_point.land_cover,
            'terrain_penalty': direct_rx_point.terrain_penalty,
            'rssi_dbm': direct_rx_point.rssi,
            'snr_db': direct_rx_point.snr,
            'pdr': direct_rx_point.pdr,
            'path_loss_db': direct_rx_point.path_loss
        })
        df_direct = pd.DataFrame(direct_data)
        direct_file = self.output_dir / 'direct_path.csv'
        df_direct.to_csv(direct_file, index=False)
        logger.info(f"  Direct path saved: {direct_file}")

        # 4. Export Model Comparison
        model_comparison = []
        for model_name, metrics in model_performances.items():
            model_comparison.append({
                'model_name': model_name,
                'rssi_mse': metrics.get('RSSI_mse', 'N/A'),
                'rssi_r2': metrics.get('RSSI_r2', 'N/A'),
                'snr_mse': metrics.get('SNR_mse', 'N/A'),
                'snr_r2': metrics.get('SNR_r2', 'N/A'),
                'path_loss_mse': metrics.get('path_loss_mse', 'N/A'),
                'path_loss_r2': metrics.get('path_loss_r2', 'N/A'),
                'average_r2': metrics.get('average_r2', 'N/A')
            })
        df_models = pd.DataFrame(model_comparison)
        models_file = self.output_dir / 'model_comparison.csv'
        df_models.to_csv(models_file, index=False)
        logger.info(f"  Model comparison saved: {models_file}")

        # 5. Export Feature Importance (if available)
        if feature_importance_data:
            df_importance = pd.DataFrame(feature_importance_data)
            importance_file = self.output_dir / 'feature_importance.csv'
            df_importance.to_csv(importance_file, index=False)
            logger.info(f"  Feature importance saved: {importance_file}")

        logger.info("All CSV exports completed!")
        
    def plot_training_history(self, train_losses, val_losses, model_name='Neural Network'):
            """Plot training and validation loss history"""
            logger.info(f"Plotting training history for {model_name}...")
            
            plt.figure(figsize=(10, 6))
            plt.plot(train_losses, label='Training Loss', linewidth=2)
            plt.plot(val_losses, label='Validation Loss', linewidth=2)
            plt.xlabel('Epoch', fontsize=12)
            plt.ylabel('Loss (MSE)', fontsize=12)
            plt.title(f'{model_name} Training History', fontsize=14, fontweight='bold')
            plt.legend(fontsize=11)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            
            filename = self.output_dir / f'{model_name.lower().replace(" ", "_")}_training_history.png'
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()
            logger.info(f"  Training history saved: {filename}")
            
    def plot_model_comparison(self, model_performances):
        """Plot model comparison bar chart"""
        logger.info("Plotting model comparison...")
        
        models = list(model_performances.keys())
        r2_scores = [model_performances[m]['average_r2'] for m in models]
        
        fig, ax = plt.subplots(figsize=(12, 6))
        bars = ax.bar(models, r2_scores, color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'])
        
        ax.set_ylabel('Average R² Score', fontsize=12)
        ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
        ax.set_ylim(0, 1.0)
        ax.grid(True, axis='y', alpha=0.3)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.4f}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        filename = self.output_dir / 'model_comparison.png'
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"  Model comparison saved: {filename}")

    def plot_feature_importance(self, feature_names, importances, model_name='Random Forest'):
        """Plot feature importance"""
        logger.info(f"Plotting feature importance for {model_name}...")
        
        # Sort by importance
        indices = np.argsort(importances)[::-1][:15]  # Top 15 features
        sorted_features = [feature_names[i] for i in indices]
        sorted_importances = [importances[i] for i in indices]
        
        plt.figure(figsize=(10, 8))
        plt.barh(range(len(sorted_features)), sorted_importances, color='steelblue')
        plt.yticks(range(len(sorted_features)), sorted_features)
        plt.xlabel('Importance', fontsize=12)
        plt.title(f'{model_name} - Top 15 Feature Importance', fontsize=14, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.grid(True, axis='x', alpha=0.3)
        plt.tight_layout()
        
        filename = self.output_dir / f'{model_name.lower().replace(" ", "_")}_feature_importance.png'
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"  Feature importance saved: {filename}")
    
    def plot_path_comparison(self, optimal_path, direct_path_points):
        """Plot comparison between optimal and direct path"""
        logger.info("Plotting path comparison...")
        
        # Prepare data
        optimal_pdr = [p.pdr for p in optimal_path]
        optimal_snr = [p.snr for p in optimal_path]
        optimal_rssi = [p.rssi for p in optimal_path]
        optimal_path_loss = [p.path_loss for p in optimal_path]
        
        direct_pdr = [p.pdr for p in direct_path_points]
        direct_snr = [p.snr for p in direct_path_points]
        direct_rssi = [p.rssi for p in direct_path_points]
        direct_path_loss = [p.path_loss for p in direct_path_points]
        
        # Create 2x2 subplot
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # PDR Comparison
        axes[0, 0].plot(optimal_pdr, 'ro-', label='Optimal Path', linewidth=2, markersize=6)
        axes[0, 0].plot(direct_pdr, 'b^--', label='Direct Path', linewidth=2, markersize=6)
        axes[0, 0].set_ylabel('PDR', fontsize=11)
        axes[0, 0].set_title('Packet Delivery Ratio (PDR)', fontsize=12, fontweight='bold')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        axes[0, 0].set_ylim(0, 1.0)
        
        # SNR Comparison
        axes[0, 1].plot(optimal_snr, 'ro-', label='Optimal Path', linewidth=2, markersize=6)
        axes[0, 1].plot(direct_snr, 'b^--', label='Direct Path', linewidth=2, markersize=6)
        axes[0, 1].set_ylabel('SNR (dB)', fontsize=11)
        axes[0, 1].set_title('Signal-to-Noise Ratio (SNR)', fontsize=12, fontweight='bold')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # RSSI Comparison
        axes[1, 0].plot(optimal_rssi, 'ro-', label='Optimal Path', linewidth=2, markersize=6)
        axes[1, 0].plot(direct_rssi, 'b^--', label='Direct Path', linewidth=2, markersize=6)
        axes[1, 0].set_xlabel('Sample/Beacon Point', fontsize=11)
        axes[1, 0].set_ylabel('RSSI (dBm)', fontsize=11)
        axes[1, 0].set_title('Received Signal Strength Indicator (RSSI)', fontsize=12, fontweight='bold')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Path Loss Comparison
        axes[1, 1].plot(optimal_path_loss, 'ro-', label='Optimal Path', linewidth=2, markersize=6)
        axes[1, 1].plot(direct_path_loss, 'b^--', label='Direct Path', linewidth=2, markersize=6)
        axes[1, 1].set_xlabel('Sample/Beacon Point', fontsize=11)
        axes[1, 1].set_ylabel('Path Loss (dB)', fontsize=11)
        axes[1, 1].set_title('Path Loss', fontsize=12, fontweight='bold')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        filename = self.output_dir / 'path_comparison.png'
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"  Path comparison saved: {filename}")
        
    def visualize_path_html(self, optimal_path, direct_path_metrics, grid_points,
                       start_lat, start_lon, dest_lat, dest_lon,
                       filename='path_visualization.html'):
        """
        Create interactive HTML map with Folium
        Shows: grid points, direct path with samples, optimal path with beacons
        """
        logger.info(f"Creating HTML visualization: {filename}")
        
        # Calculate center
        all_lats = [p.lat for p in grid_points]
        all_lons = [p.lon for p in grid_points]
        center_lat = np.mean(all_lats)
        center_lon = np.mean(all_lons)
        
        # Create map
        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=13,
            tiles='OpenStreetMap'
        )
        
        # Add grid points as background
        for point in grid_points:
            actual_pdr = point.pdr if point.pdr > 0 else 0.0
            color = self._get_color_for_pdr(actual_pdr)
            folium.CircleMarker(
                location=[point.lat, point.lon],
                radius=3,
                popup=\
                    f"Grid Point<br>"
                    f"PDR: {point.pdr:.3f}<br>"
                    f"RSSI: {point.rssi:.1f} dBm<br>"
                    f"SNR: {point.snr:.1f} dB<br>"
                    f"Land Cover: {point.land_cover}",
                color=color,
                fill=True,
                fill_opacity=0.5
            ).add_to(m)
        
        # ========================================================================
        # DIRECT PATH VISUALIZATION WITH SAMPLE POINTS
        # ========================================================================
        direct_path_points = direct_path_metrics.get('points', [])
        
        if direct_path_points:
            # Build direct path coordinates (transmitter → samples → receiver)
            direct_coords = [[start_lat, start_lon]]
            direct_coords.extend([[p.lat, p.lon] for p in direct_path_points])
            direct_coords.append([dest_lat, dest_lon])
            
            # Draw direct path polyline
            folium.PolyLine(
                direct_coords,
                color='blue',
                weight=3,
                opacity=0.7,
                dash_array='10',
                popup=f"<b>Direct Path</b><br>"
                    f"Avg PDR: {direct_path_metrics['PDR']:.3f} ({direct_path_metrics['PDR']*100:.1f}%)<br>"
                    f"Avg RSSI: {direct_path_metrics['RSSI']:.1f} dBm<br>"
                    f"Avg SNR: {direct_path_metrics['SNR']:.2f} dB<br>"
                    f"Sample Points: {len(direct_path_points)}"
            ).add_to(m)
            
            # Add sample point markers on direct path
            for i, point in enumerate(direct_path_points):
                folium.CircleMarker(
                    location=[point.lat, point.lon],
                    radius=5,
                    popup=f"<b>Direct Path Sample {i+1}</b><br>"
                        f"PDR: {point.pdr:.3f}<br>"
                        f"RSSI: {point.rssi:.1f} dBm<br>"
                        f"SNR: {point.snr:.1f} dB<br>"
                        f"Elevation: {point.elevation:.0f}m<br>"
                        f"Land Cover: {point.land_cover}",
                    color='blue',
                    fill=True,
                    fill_color='lightblue',
                    fill_opacity=0.7,
                    weight=2
                ).add_to(m)
        else:
            # Fallback: simple direct line if no sample points
            direct_coords = [[start_lat, start_lon], [dest_lat, dest_lon]]
            folium.PolyLine(
                direct_coords,
                color='blue',
                weight=3,
                opacity=0.7,
                dash_array='10',
                popup=f"Direct Path<br>Avg PDR: {direct_path_metrics['PDR']:.3f}"
            ).add_to(m)
        
        # ========================================================================
        # OPTIMAL PATH VISUALIZATION
        # ========================================================================
        # Build complete path: transmitter → beacons → receiver
        complete_path_coords = [[start_lat, start_lon]]
        complete_path_coords.extend([[p.lat, p.lon] for p in optimal_path])
        complete_path_coords.append([dest_lat, dest_lon])
        
        # Draw connected optimal path
        avg_optimal_pdr = np.mean([p.pdr for p in optimal_path]) if optimal_path else 0.0
        folium.PolyLine(
            complete_path_coords,
            color='red',
            weight=4,
            opacity=0.9,
            popup=f"<b>Optimal Path</b><br>"
                f"Beacons: {len(optimal_path)}<br>"
                f"Avg PDR: {avg_optimal_pdr:.3f} ({avg_optimal_pdr*100:.1f}%)<br>"
                f"Min PDR: {min([p.pdr for p in optimal_path]):.3f}"
        ).add_to(m)
        
        # Add beacon markers
        for i, point in enumerate(optimal_path):
            folium.Marker(
                location=[point.lat, point.lon],
                popup=f"<b>Beacon {i+1}</b><br>"
                    f"PDR: {point.pdr:.3f}<br>"
                    f"RSSI: {point.rssi:.1f} dBm<br>"
                    f"SNR: {point.snr:.1f} dB<br>"
                    f"Elevation: {point.elevation:.0f}m<br>"
                    f"Land Cover: {point.land_cover}",
                icon=folium.Icon(color='red', icon='info-sign')
            ).add_to(m)
        
        # ========================================================================
        # START AND END MARKERS
        # ========================================================================
        # Add transmitter marker
        folium.Marker(
            location=[start_lat, start_lon],
            popup="<b>Transmitter</b><br>(Start Point)",
            icon=folium.Icon(color='green', icon='play', prefix='fa')
        ).add_to(m)
        
        # Add receiver marker
        folium.Marker(
            location=[dest_lat, dest_lon],
            popup="<b>Receiver</b><br>(Destination)",
            icon=folium.Icon(color='green', icon='stop', prefix='fa')
        ).add_to(m)
        
        # ========================================================================
        # LEGEND
        # ========================================================================
        legend_html = '''
        <div style="position: fixed; bottom: 50px; left: 50px; width: 250px; height: 180px; 
                    background-color:white; border:2px solid grey; z-index:9999; 
                    font-size:14px; padding: 10px">
        <p><strong>Path Visualization Legend</strong></p>
        <p><i class="fa fa-minus" style="color:blue"></i> Direct Path (dashed) + samples</p>
        <p><i class="fa fa-minus" style="color:red"></i> Optimal Path (solid)</p>
        <p><i class="fa fa-map-marker" style="color:green"></i> Transmitter/Receiver</p>
        <p><i class="fa fa-map-marker" style="color:red"></i> Relay Beacons</p>
        <p><i class="fa fa-circle" style="color:lightblue"></i> Direct Path Samples</p>
        <p><i class="fa fa-circle" style="color:lightgray"></i> Grid Points (background)</p>
        </div>
        '''
        m.get_root().html.add_child(folium.Element(legend_html))
        
        # ========================================================================
        # SAVE MAP
        # ========================================================================
        filepath = self.output_dir / filename
        try:
            m.save(str(filepath))
            logger.info(f"HTML map saved to {filepath}")
        except Exception as e:
            logger.error(f"Error saving map: {e}")
            m.save(filename)

    def _get_color_for_pdr(self, pdr):
        """Get color based on PDR value"""
        if pdr >= 0.9:
            return 'green'
        elif pdr >= 0.7:
            return 'lightgreen'
        elif pdr >= 0.5:
            return 'yellow'
        elif pdr >= 0.3:
            return 'orange'
        else:
            return 'red'
    
    def print_path_summary(self, optimal_path, direct_path):
        """Print comprehensive path summary"""
        print("="*70)
        print("PATH OPTIMIZATION SUMMARY")
        print("="*70)
        
        opt_avg_rssi = np.mean([p.rssi for p in optimal_path])
        opt_avg_snr = np.mean([p.snr for p in optimal_path])
        opt_avg_pdr = np.mean([p.pdr for p in optimal_path])
        opt_min_pdr = min([p.pdr for p in optimal_path])
        opt_avg_elevation = np.mean([p.elevation for p in optimal_path])
        opt_avg_terrain = np.mean([p.terrain_penalty for p in optimal_path])
        
        dir_rssi = direct_path['RSSI']
        dir_snr = direct_path['SNR']
        dir_pdr = direct_path['PDR']
        
        print(f"Direct Path:")
        print(f"  Average RSSI: {dir_rssi:.2f} dBm")
        print(f"  Average SNR:  {dir_snr:.2f} dB")
        print(f"  Average PDR:  {dir_pdr:.4f} ({dir_pdr*100:.2f}%)")
        
        print(f"Optimal Path:")
        print(f"  Average RSSI: {opt_avg_rssi:.2f} dBm")
        print(f"  Average SNR:  {opt_avg_snr:.2f} dB")
        print(f"  Average PDR:  {opt_avg_pdr:.4f} ({opt_avg_pdr*100:.2f}%)")
        print(f"  Minimum PDR:  {opt_min_pdr:.4f} ({opt_min_pdr*100:.2f}%)")
        print(f"  Path length:  {len(optimal_path)} beacons")
        print(f"  Avg Elevation: {opt_avg_elevation:.1f} m (from SRTM)")
        print(f"  Avg Terrain : {opt_avg_terrain:.3f} (from ESA WorldCover)")
        print(f"  Avg Terrain Penalty: {opt_avg_terrain:.3f} (from ESA WorldCover)")
        
        print(f"Improvements:")
        rssi_imp = opt_avg_rssi - dir_rssi
        snr_imp = opt_avg_snr - dir_snr
        pdr_imp = (opt_avg_pdr - dir_pdr) * 100
        
        print(f"  RSSI: {rssi_imp:+.2f} dBm ({rssi_imp/abs(dir_rssi)*100:+.2f}%)")
        print(f"  SNR:  {snr_imp:+.2f} dB ({snr_imp/abs(dir_snr)*100:+.2f}%)")
        print(f"  PDR:  {pdr_imp:+.2f}%")
        print("="*70 + "")


### Main System Integration

In [16]:
class ImprovedLoRaSystem:
    """Complete LoRa optimization system"""
    def __init__(self, config_dict=None):
        """Initialize system with configuration"""
        self.config = config_dict or self._default_config()
        self.device = device
        
        logger.info("="*70)
        logger.info("INITIALIZING IMPROVED LORA SYSTEM")
        logger.info("="*70)
        logger.info(f"Device: {self.device}")
        
        # Initialize components
        gee_config = GEEConfig(**self.config['gee'])
        self.gee = BatchGEEIntegration(gee_config)
        self.preprocessor = LoRaDataPreprocessor(gee_integration=self.gee)
        self.visualizer = ResultVisualizer()
        self.physics_engine = LoRaPhysicsEngine()
        
        # Model storage
        self.models = {}
        self.scalers = {}
        self.best_model_name = None
    
    def _default_config(self):
        """Default configuration"""
        return {
            'data': {
                'dataset1_path': r'../data/processed_data_1.csv',
                'dataset2_path': r'../data/processed_data_2.csv',
                'test_size': 0.2,
                'random_state': 42
            },
            
            # Hyperparameter tuning configuration
            'hyperparameter_tuning': {
                'enable': False,              # Set to True to enable tuning
                'nn_trials': 40,             # Number of trials for neural network
                'rf_n_iter': 40,             # Number of iterations for Random Forest
                'xgb_n_iter': 40,            # Number of iterations for XGBoost
                'cv_folds': 3,               # Number of cross-validation folds
                'tuning_data_ratio': 0.2     # Portion of training data to use for tuning
            },
            
            # Model hyperparameters (used only if hyperparameter tuning is disabled)
            'model_hyperparams': {
                'neural_network': {
                    'hidden_sizes': [256, 128, 64, 32],
                    'dropout_rate': 0.3,
                    'activation': 'relu',
                    'batch_size': 64,
                    'learning_rate': 0.001,
                    'weight_decay': 1e-5,
                    'epochs': 400,
                    'early_stopping_patience': 20,
                    'gradient_clip': 1.0
                },
                'random_forest': {
                    'n_estimators': 100,
                    'max_depth': None,
                    'min_samples_split': 2,
                    'min_samples_leaf': 1,
                    'max_features': None
                },
                'xgboost': {
                    'n_estimators': 100,
                    'learning_rate': 0.1,
                    'max_depth': 6,
                    'subsample': 1.0,
                    'colsample_bytree': 1.0,
                    'min_child_weight': 1
                }
            },
            
            'gee': {
                'batch_size': 100,
                'workers': 5,
                'retry_attempts': 3,
                'fallback_to_individual': True,
                'cache_enabled': True,
                'cache_file': 'gee_cache.pkl',
                'path_spatial_samples': 10
            },
            
            'optimization': {
                'grid_spacing_km': 1.5,
                'corridor_width_km': 4.0,
                'adaptive_grid': True,
                'max_path_deviation': 0.5,
                'min_pdr_threshold': 0.3,
                'prefer_water': True,
                'avoid_buildings': True
            }
        }
    
    def load_and_preprocess_data(self):
        """Load and preprocess datasets"""
        logger.info("Loading and preprocessing data...")
        data_config = self.config['data']
        
        datasets = []
        
        for path_key in ['dataset1_path', 'dataset2_path']:
            if path_key in data_config and os.path.exists(data_config[path_key]):
                try:
                    df = self.preprocessor.load_dataset(data_config[path_key])
                    logger.info(f"  Dataset loaded: {len(df)} rows")
                    datasets.append(df)
                except Exception as e:
                    logger.warning(f"  Could not load dataset: {e}")
        
        if not datasets:
            raise ValueError("No datasets could be loaded!")
        
        df_combined = self.preprocessor.merge_datasets(*datasets)
        X_train, X_test, y_train, y_test, feature_cols = self.preprocessor.prepare_features(df_combined)
        
        return X_train, X_test, y_train, y_test, feature_cols

    def train_models_and_select_best(self, X_train, X_test, y_train, y_test, feature_cols):
        """Train all models and auto-select best"""
        logger.info("="*70)
        logger.info("TRAINING ALL MODELS")
        logger.info("="*70)
        
        tuning_config = HyperparameterConfig(**self.config['hyperparameter_tuning'])
        
        # Split data for tuning if enabled
        if tuning_config.enable:
            logger.info("HYPERPARAMETER TUNING: ENABLED")
            logger.info(f"  NN trials: {tuning_config.nn_trials}")
            logger.info(f"  RF iterations: {tuning_config.rf_n_iter}")
            logger.info(f"  XGB iterations: {tuning_config.xgb_n_iter}")
            logger.info(f"  CV folds: {tuning_config.cv_folds}")
            
            # Split training data for tuning
            split_idx = int(len(X_train) * (1 - tuning_config.tuning_data_ratio))
            X_train_tune = X_train[:split_idx]
            y_train_tune = y_train[:split_idx]
            X_val_tune = X_train[split_idx:]
            y_val_tune = y_train[split_idx:]
            
            logger.info(f"  Tuning data: {len(X_train_tune)} train, {len(X_val_tune)} val")
        else:
            logger.info("HYPERPARAMETER TUNING: DISABLED (using default hyperparameters)")
        
        # Initialize selector
        selector = BestModelSelector(self.device)
        
        # ========================================================================
        # 1. NEURAL NETWORK
        # ========================================================================
        logger.info("" + "="*70)
        logger.info("1. TRAINING NEURAL NETWORK")
        logger.info("="*70)
        
        if tuning_config.enable and OPTUNA_AVAILABLE:
            # Tune hyperparameters
            nn_tuner = NeuralNetworkTuner(
                X_train_tune, y_train_tune, X_val_tune, y_val_tune,
                self.device, tuning_config.nn_trials
            )
            nn_best_params = nn_tuner.tune()
            
            # Build config from tuned params
            nn_config = {
                'hidden_sizes': nn_best_params['hidden_sizes'],
                'dropout_rate': nn_best_params['dropout_rate'],
                'activation': nn_best_params['activation'],
                'batch_norm': True
            }
            training_config = {
                'batch_size': nn_best_params['batch_size'],
                'learning_rate': nn_best_params['learning_rate'],
                'weight_decay': nn_best_params['weight_decay'],
                'epochs': 400,
                'early_stopping_patience': 20,
                'gradient_clip': 1.0,
                'model': nn_config
            }
            logger.info("  Using TUNED hyperparameters")
        else:
            # Use default hyperparameters
            nn_params = self.config['model_hyperparams']['neural_network']
            nn_config = {
                'hidden_sizes': nn_params['hidden_sizes'],
                'dropout_rate': nn_params['dropout_rate'],
                'activation': nn_params['activation'],
                'batch_norm': True
            }
            training_config = {
                'batch_size': nn_params['batch_size'],
                'learning_rate': nn_params['learning_rate'],
                'weight_decay': nn_params['weight_decay'],
                'epochs': nn_params['epochs'],
                'early_stopping_patience': nn_params['early_stopping_patience'],
                'gradient_clip': nn_params['gradient_clip'],
                'model': nn_config
            }
            logger.info("  Using DEFAULT hyperparameters")
        
        # Train Neural Network
        train_dataset = LoRaDataset(X_train, y_train)
        test_dataset = LoRaDataset(X_test, y_test)
        train_loader = DataLoader(train_dataset, batch_size=training_config['batch_size'], shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=training_config['batch_size'], shuffle=False)
        
        nn_trainer = NeuralNetworkTrainer(
            input_size=X_train.shape[1],
            output_size=y_train.shape[1],
            device=self.device,
            config=training_config
        )
        nn_trainer.train(train_loader, test_loader)
        selector.add_model('Neural_Network', nn_trainer.model)
        
        # ========================================================================
        # 2. RANDOM FOREST
        # ========================================================================
        logger.info("" + "="*70)
        logger.info("2. TRAINING RANDOM FOREST")
        logger.info("="*70)
        
        if tuning_config.enable:
            # Tune hyperparameters
            rf_tuner = RandomForestTuner(
                X_train_tune, y_train_tune,
                tuning_config.rf_n_iter, tuning_config.cv_folds
            )
            rf_best_params = rf_tuner.tune()
            rf_model = RandomForestModel(**rf_best_params)
            logger.info("  Using TUNED hyperparameters")
        else:
            # Use default hyperparameters
            rf_params = self.config['model_hyperparams']['random_forest']
            rf_model = RandomForestModel(**rf_params)
            logger.info("  Using DEFAULT hyperparameters")
        
        rf_model.train(X_train, y_train)
        selector.add_model('Random_Forest', rf_model)
        
        # ========================================================================
        # 3. XGBOOST
        # ========================================================================
        logger.info("" + "="*70)
        logger.info("3. TRAINING XGBOOST")
        logger.info("="*70)
        
        if tuning_config.enable:
            # Tune hyperparameters
            xgb_tuner = XGBoostTuner(
                X_train_tune, y_train_tune,
                tuning_config.xgb_n_iter, tuning_config.cv_folds
            )
            xgb_best_params = xgb_tuner.tune()
            xgb_model = XGBoostModel(**xgb_best_params)
            logger.info("  Using TUNED hyperparameters")
        else:
            # Use default hyperparameters
            xgb_params = self.config['model_hyperparams']['xgboost']
            xgb_model = XGBoostModel(**xgb_params)
            logger.info("  Using DEFAULT hyperparameters")
        
        xgb_model.train(X_train, y_train)
        selector.add_model('XGBoost', xgb_model)
        
        # ========================================================================
        # 4. EVALUATE ALL MODELS
        # ========================================================================
        selector.evaluate_all(X_test, y_test)
        
        selector.print_comparison_table()
        # ========================================================================
        # 5. CREATE ENSEMBLE
        # ========================================================================
        selector.create_ensemble(X_test, y_test)
        
        selector.print_comparison_table()
        # ========================================================================
        # 6. SELECT BEST MODEL
        # ========================================================================
        best_model, best_name = selector.select_best()
        
        # ========================================================================
        # 7. SAVE BEST MODEL
        # ========================================================================
        selector.save_best_model(self.preprocessor.scaler, feature_cols)
        
        selector.print_accuracy_interpretation()
        
        # Store in system
        self.models['best'] = best_model
        self.best_model_name = best_name
        self.scalers['feature'] = self.preprocessor.scaler
        
        # Store selector and feature_cols for later use
        self.selector = selector
        self.feature_cols = feature_cols
        
        logger.info(f"  Best model selected: {best_name}")
        
        # Plot training history for Neural Network
        if hasattr(nn_trainer, 'train_losses'):
            self.visualizer.plot_training_history(
                nn_trainer.train_losses,
                nn_trainer.val_losses,
                model_name='Neural Network'
            )
            
        return best_model, best_name
    
    def predict_and_optimize(self, start_lat, start_lon, dest_lat, dest_lon,
                           spreading_factor=7, tx_power=14, frequency=868,
                           grid_spacing_km=1.5, gee_workers=5,
                           corridor_width_km=4.0, adaptive_grid=True,
                           max_path_deviation=0.5, min_pdr_threshold=0.3,
                           prefer_water=True, avoid_buildings=True,
                           direct_path_threshold_km=1.0,direct_path_use_grid=True,
                           direct_path_samples=None,direct_path_spacing_km=None):
        """Main prediction and optimization function"""
        logger.info("PREDICTION AND OPTIMIZATION")
        logger.info("="*70)
        
        # Validate inputs
        try:
            # Validate coordinates
            validate_coordinates(start_lat, start_lon, "Start")
            validate_coordinates(dest_lat, dest_lon, "Destination")
            validate_distance(start_lat, start_lon, dest_lat, dest_lon)
            
            # Validate LoRa parameters
            validate_lora_parameters(spreading_factor, tx_power, frequency)
            
            # Validate grid parameters
            validate_grid_parameters(grid_spacing_km, corridor_width_km, adaptive_grid)
            
            # Validate GEE parameters
            validate_gee_parameters(gee_workers)
            
            # Validate optimization parameters
            validate_optimization_parameters(
                max_path_deviation, min_pdr_threshold,
                prefer_water, avoid_buildings,
                direct_path_threshold_km
            )
            
            # Validate direct path parameters
            validate_direct_path_parameters(
                direct_path_use_grid, 
                direct_path_samples, 
                direct_path_spacing_km
            )
            logger.info("All parameters validated successfully")
            
        except (InvalidCoordinatesError, InvalidLoRaParametersError, ValueError) as e:
            logger.error(f"INPUT VALIDATION FAILED:")
            logger.error(f"  {str(e)}")
            logger.error(f"Please check your parameters and try again.")
            raise
        
        # Create LoRa parameters
        lora_params = LoRaParameters(
            tx_power=tx_power,
            spreading_factor=spreading_factor,
            frequency=frequency
        )
        
        # Calculate distance
        R = 6371000
        phi1, phi2 = np.radians(start_lat), np.radians(dest_lat)
        dphi = np.radians(dest_lat - start_lat)
        dlambda = np.radians(dest_lon - start_lon)
        a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        distance_m = R * c
        distance_km = distance_m / 1000
        
        logger.info(f"Distance: {distance_km:.2f} km")
        
        # Update GEE workers
        self.gee.config.workers = gee_workers
        
        # Create optimization config
        opt_config = OptimizationConfig(
            grid_spacing_km=grid_spacing_km,
            corridor_width_km=corridor_width_km,
            adaptive_grid=adaptive_grid,
            max_path_deviation=max_path_deviation,
            min_pdr_threshold=min_pdr_threshold,
            prefer_water=prefer_water,
            avoid_buildings=avoid_buildings
        )
        
        # Check if model is trained
        if 'best' not in self.models:
            raise ValueError("No trained model available. Please run train_models_and_select_best() first.")
        
        logger.info(f"Using BEST model: {self.best_model_name}")
        
        # Create universal model wrapper
        class UniversalModelWrapper:
            def __init__(self, model, model_name, device):
                self.model = model
                self.model_name = model_name
                self.device = device
            
            def predict(self, X):
                if 'Neural' in self.model_name or hasattr(self.model, 'eval'):
                    self.model.eval()
                    with torch.no_grad():
                        X_tensor = torch.FloatTensor(X).to(self.device)
                        return self.model(X_tensor).cpu().numpy()
                else:
                    return self.model.predict(X)
        
        wrapper = UniversalModelWrapper(self.models['best'], self.best_model_name, self.device)
        
        # Create optimizer
        optimizer = PathOptimizer(
            wrapper,
            self.scalers['feature'],
            [],
            self.gee,
            opt_config
        )
        
        # SHORT DISTANCE: Use direct path
        if distance_km < direct_path_threshold_km:
            logger.info(f"SHORT DISTANCE ({distance_km:.2f} km < {direct_path_threshold_km} km)")
            logger.info("Using DIRECT PATH")
            
            direct_link = optimizer.predict_hop(
                start_lat, start_lon, dest_lat, dest_lon, lora_params
            )
            
            logger.info(f"Direct Link Quality:")
            logger.info(f"  RSSI: {direct_link.rssi:.1f} dBm")
            logger.info(f"  SNR: {direct_link.snr:.2f} dB")
            logger.info(f"  PDR: {direct_link.pdr:.3f} ({direct_link.pdr*100:.1f}%)")
            
            if direct_link.pdr >= min_pdr_threshold:
                logger.info(f"  Direct link is VIABLE")
                
                result = {
                    'route': [
                        {'lat': start_lat, 'lon': start_lon, 'type': 'transmitter'},
                        {'lat': dest_lat, 'lon': dest_lon, 'type': 'receiver'}
                    ],
                    'metrics': {
                        'avg_pdr': float(direct_link.pdr),
                        'min_pdr': float(direct_link.pdr),
                        'avg_rssi': float(direct_link.rssi),
                        'avg_snr': float(direct_link.snr),
                        'num_beacons': 0,
                        'model_used': self.best_model_name,
                        'routing_mode': 'direct'
                    },
                    'comparison': {
                        'direct_path_pdr': float(direct_link.pdr),
                        'optimal_path_pdr': float(direct_link.pdr),
                        'improvement_percent': 0.0
                    }
                }
                
                return result
        
        # LONG DISTANCE: Use A* optimization
        logger.info("Using A* OPTIMIZATION with beacons")
        
        optimal_path, grid_points, optimal_rx_point = optimizer.find_optimal_path(
            start_lat, start_lon, dest_lat, dest_lon, lora_params, opt_config
        )

        
        # Sample direct path with user-specified options
        direct_path_metrics = optimizer.sample_direct_path(
            start_lat, start_lon, dest_lat, dest_lon,
            lora_params,
            use_grid_alignment=direct_path_use_grid,
            num_samples=direct_path_samples,
            beacon_spacing_km=direct_path_spacing_km
        )
        
        direct_rx_point = direct_path_metrics['rx_point']
        
        self.visualizer.visualize_path_html(
            optimal_path, direct_path_metrics, grid_points,
            start_lat, start_lon, dest_lat, dest_lon
        )
        
        self.visualizer.print_path_summary(optimal_path, direct_path_metrics)
        
        result = {
            'route': [
                {'lat': start_lat, 'lon': start_lon, 'type': 'transmitter'}
            ] + [
                {
                    'lat': p.lat,
                    'lon': p.lon,
                    'type': 'beacon',
                    'pdr': float(p.pdr),
                    'rssi': float(p.rssi),
                    'snr': float(p.snr),
                    'elevation': float(p.elevation),
                    'land_cover': int(p.land_cover)
                }
                for p in optimal_path
            ] + [
                {'lat': dest_lat, 'lon': dest_lon, 'type': 'receiver'}
            ],
            'metrics': {
                'avg_pdr': float(np.mean([p.pdr for p in optimal_path if p.pdr > 0])),
                'min_pdr': float(min([p.pdr for p in optimal_path if p.pdr > 0])),
                'avg_rssi': float(np.mean([p.rssi for p in optimal_path])),
                'avg_snr': float(np.mean([p.snr for p in optimal_path])),
                'num_beacons': len(optimal_path),
                'model_used': self.best_model_name,
                'routing_mode': 'optimized'
            },
            'comparison': {
                'direct_path_pdr': float(direct_path_metrics['PDR']),
                'optimal_path_pdr': float(np.mean([p.pdr for p in optimal_path if p.pdr > 0])),
                'improvement_percent': float(
                    ((np.mean([p.pdr for p in optimal_path if p.pdr > 0]) - direct_path_metrics['PDR']) 
                     / direct_path_metrics['PDR']) * 100
                )
            },
            'files': {
                'map': str(self.visualizer.output_dir / 'path_visualization.html')
            }
        }
        
        logger.info("Optimization completed successfully!")
        
        # ============================================================
        # EXPORT CSV AND GENERATE PLOTS
        # ============================================================
        logger.info("="*70)
        logger.info("GENERATING EXPORTS AND VISUALIZATIONS")
        logger.info("="*70)
        
        # Get feature importance
        if hasattr(self, 'selector'):
            feature_importance_data = self.selector.get_feature_importance(
                self.feature_cols  # You need to store feature_cols
            )
            
            if feature_importance_data:
                # Save feature importance CSV
                df_importance = pd.DataFrame(feature_importance_data)
                importance_file = self.visualizer.output_dir / 'feature_importance.csv'
                df_importance.to_csv(importance_file, index=False)
                logger.info(f"Feature importance saved: {importance_file}")
                
                # Plot feature importance
                self.visualizer.plot_feature_importance(
                    feature_importance_data['feature'],
                    feature_importance_data['importance'],
                    model_name=self.best_model_name
                )

        # Export CSVs
        self.visualizer.export_results_to_csv(
            optimal_path=optimal_path,
            grid_points=grid_points,
            direct_path_points=direct_path_metrics['points'],
            model_performances=self.selector.performances,
            optimal_rx_point=optimal_rx_point,
            direct_rx_point=direct_rx_point,
            feature_importance_data=feature_importance_data
        )
        
        # Plot model comparison
        if hasattr(self, 'selector'):
            self.visualizer.plot_model_comparison(self.selector.performances)
        
        # Plot path comparison
        self.visualizer.plot_path_comparison(optimal_path, direct_path_metrics['points'])
        
        logger.info("All exports and visualizations completed!")
        
        return result


### Example Usage

In [17]:
if __name__ == "__main__":
    
    # ============================================================================
    # CONFIGURATION - CUSTOMIZE ALL PARAMETERS HERE
    # ============================================================================
    
    CONFIG = {
        # Data loading configuration
        'data': {
            'dataset1_path': r'../data/processed_data_1.csv',
            'dataset2_path': r'../data/processed_data_2.csv',
            'test_size': 0.2,
            'random_state': 42
        },
        
        # Hyperparameter tuning configuration
        'hyperparameter_tuning': {
            'enable': True,              # Set to True to enable tuning
            'nn_trials': 100,             # Number of trials for neural network
            'rf_n_iter': 500,             # Number of iterations for Random Forest
            'xgb_n_iter': 500,            # Number of iterations for XGBoost
            'cv_folds': 5,               # Number of cross-validation folds
            'tuning_data_ratio': 0.25     # Portion of training data to use for tuning
        },
        
        # Model hyperparameters (used only if hyperparameter tuning is disabled)
        'model_hyperparams': {
            'neural_network': {
                'hidden_sizes': [256, 128, 64, 32],
                'dropout_rate': 0.3,
                'activation': 'relu',
                'batch_size': 64,
                'learning_rate': 0.001,
                'weight_decay': 1e-5,
                'epochs': 400,
                'early_stopping_patience': 20,
                'gradient_clip': 1.0
            },
            'random_forest': {
                'n_estimators': 200,
                'max_depth': None,
                'min_samples_split': 2,
                'min_samples_leaf': 1,
                'max_features': None
            },
            'xgboost': {
                'n_estimators': 200,
                'learning_rate': 0.1,
                'max_depth': 6,
                'subsample': 1.0,
                'colsample_bytree': 1.0,
                'min_child_weight': 1
            }
        },
        
        # Google Earth Engine configuration
        'gee': {
            'batch_size': 100,
            'workers': 8,
            'retry_attempts': 3,
            'fallback_to_individual': True,
            'cache_enabled': True,
            'cache_file': 'gee_cache.pkl',
            'path_spatial_samples': 15
        },
        
        # Path optimization configuration
        'optimization': {
            'grid_spacing_km': 1.5,
            'corridor_width_km': 4.0,
            'adaptive_grid': False,
            'max_path_deviation': 0.5,
            'min_pdr_threshold': 0.3,
            'prefer_water': True,
            'avoid_buildings': True
        }
    }
    
    # ============================================================================
    # INITIALIZE SYSTEM
    # ============================================================================
    
    system = ImprovedLoRaSystem(config_dict=CONFIG)
    
    # ============================================================================
    # LOAD DATA AND TRAIN MODELS
    # ============================================================================
    
    logger.info("="*70)
    logger.info("STEP 1: LOADING DATA")
    logger.info("="*70)
    
    X_train, X_test, y_train, y_test, feature_cols = system.load_and_preprocess_data()
    
    logger.info("="*70)
    logger.info("STEP 2: TRAINING MODELS")
    logger.info("="*70)
    
    # Train all models and auto-select best
    best_model, best_name = system.train_models_and_select_best(
        X_train, X_test, y_train, y_test, feature_cols
    )
    
    # ============================================================================
    # PREDICTION AND OPTIMIZATION EXAMPLES
    # ============================================================================
    
    logger.info("="*70)
    logger.info("STEP 3: RUNNING OPTIMIZATION EXAMPLES")
    logger.info("="*70)
    
    try:
        result_test = system.predict_and_optimize(
            # Coordinates (REQUIRED)
            # lat :-90 to 90, lon :-180 to 180
            start_lat=51.5000, start_lon=-0.1200,
            dest_lat=51.7000, dest_lon=0.1400,
            
            # LoRa Parameters (REQUIRED)
            # 7-12 (higher = longer range, slower)
            spreading_factor=7,       
            # 2-30 dBm (higher = better signal, more power)
            tx_power=14,              
            # 100-1000 MHz (EU: 868, US: 915, AS: 923)
            frequency=868,            
            
            # Grid Configuration (OPTIONAL)
            grid_spacing_km=1.0,       # 0.1-10.0 km (1.0-2.0 km recommended)
            corridor_width_km=6.0,     # 0.5-20.0 km (3.0-6.0 km recommended)
            # adaptive_grid True/False (adjust grid density based on distance)
            adaptive_grid=True,        # Auto-adjust based on distance
            
            # GEE Configuration (OPTIONAL)
            gee_workers=5,             # 1-20 (5-10 for best speed/stability)
            
            # Optimization Preferences (OPTIONAL)
            max_path_deviation=1.0,    # 0.0-3.0 (0.3-1.0 recommended)
            min_pdr_threshold=0.5,     # 0.1-1.0 (0.2-0.5 recommended)
            # True/False (water = best RF, Buildings = worst RF)
            prefer_water=True,         # Water = best RF propagation
            avoid_buildings=True,      # Buildings = worst RF propagation
            direct_path_threshold_km= 0.5,  # 0.1-10 km (0.5-2.0 km recommended)
            
            # Direct Path Sampling (OPTIONAL)
            direct_path_use_grid=True,      # True=match grid, False=custom
            # direct_path_samples=15,        # Fixed number (if use_grid=False)
            # direct_path_spacing_km=2.0,    # Custom spacing (if use_grid=False)
        )
        
        logger.info("RESULT:")
        logger.info(f"  Model used: {result_test['metrics']['model_used']}")
        logger.info(f"  Beacons needed: {result_test['metrics']['num_beacons']}")
        logger.info(f"  Minimum PDR: {result_test['metrics']['min_pdr']:.3f}")
        logger.info(f"  Average SNR: {result_test['metrics']['avg_snr']:.2f} dB")
        logger.info(f"  Average RSSI: {result_test['metrics']['avg_rssi']:.1f} dBm")
        logger.info(f"  Average PDR: {result_test['metrics']['avg_pdr']:.3f}")
        logger.info(f"  Improvement: {result_test['comparison']['improvement_percent']:.1f}%")
        logger.info(f"  Average elevation: {result_test['route'][-2]['elevation']:.1f} m")
        logger.info(f"  Average land cover: {result_test['route'][-2]['land_cover']}")
        logger.info(f"  Map saved: {result_test['files']['map']}")
        
    except Exception as e:
        logger.error(f"Example failed: {e}")
    
    # ============================================================================
    # SAVE ALL RESULTS
    # ============================================================================
    
    logger.info("" + "="*70)
    logger.info("SAVING RESULTS")
    logger.info("="*70)
    
    all_results = {
        'example_test': result_test,
    }
    
    output_file = Path("./output/optimization_results.json")
    output_file.parent.mkdir(exist_ok=True, parents=True)
    
    with open(output_file, 'w') as f:
        json.dump(all_results, f, indent=2)
    
    logger.info(f"  All results saved to: {output_file}")
    

2025-10-19 05:47:07,423 - __main__ - INFO - ======================================================================
2025-10-19 05:47:07,425 - __main__ - INFO - INITIALIZING IMPROVED LORA SYSTEM
2025-10-19 05:47:07,425 - __main__ - INFO - ======================================================================
2025-10-19 05:47:07,426 - __main__ - INFO - Device: cuda
2025-10-19 05:47:07,456 - __main__ - INFO - Loaded 82481 cached GEE results
2025-10-19 05:47:12,306 - __main__ - INFO - Google Earth Engine initialized successfully
2025-10-19 05:47:12,313 - __main__ - INFO - ======================================================================
2025-10-19 05:47:12,315 - __main__ - INFO - STEP 1: LOADING DATA
2025-10-19 05:47:12,316 - __main__ - INFO - ======================================================================
2025-10-19 05:47:12,316 - __main__ - INFO - Loading and preprocessing data...
2025-10-19 05:47:12,365 - __main__ - INFO -   Dataset loaded: 1268 rows
2025-10-19 05:47:12,379 -

  0%|          | 0/100 [00:00<?, ?it/s]

2025-10-19 05:47:12,445 - __main__ - INFO - Training Neural Network on cuda...
2025-10-19 05:47:13,627 - __main__ - INFO - Epoch [10/100] - Train Loss: 5005.925890, Val Loss: 4603.239583
2025-10-19 05:47:14,087 - __main__ - INFO - Epoch [20/100] - Train Loss: 667.786472, Val Loss: 210.774618
2025-10-19 05:47:14,466 - __main__ - INFO - Epoch [30/100] - Train Loss: 541.508192, Val Loss: 159.725418
2025-10-19 05:47:14,836 - __main__ - INFO - Epoch [40/100] - Train Loss: 487.321760, Val Loss: 140.644547
2025-10-19 05:47:15,233 - __main__ - INFO - Epoch [50/100] - Train Loss: 424.427765, Val Loss: 121.240194
2025-10-19 05:47:15,589 - __main__ - INFO - Epoch [60/100] - Train Loss: 401.865526, Val Loss: 118.546862
2025-10-19 05:47:15,921 - __main__ - INFO - Epoch [70/100] - Train Loss: 382.438161, Val Loss: 114.957759
2025-10-19 05:47:16,241 - __main__ - INFO - Epoch [80/100] - Train Loss: 351.795115, Val Loss: 105.196086
2025-10-19 05:47:16,578 - __main__ - INFO - Epoch [90/100] - Train Loss

[I 2025-10-19 05:47:16,916] Trial 0 finished with value: 101.92406972249348 and parameters: {'n_layers': 3, 'hidden_size_base': 64, 'decay_strategy': 'constant', 'dropout_rate': 0.36066900704592525, 'weight_decay': 0.000133112160807369, 'activation': 'leaky_relu', 'normalization': 'none', 'optimizer_name': 'adam', 'learning_rate': 0.000684792009557478, 'batch_size': 256, 'gradient_clip': 4.033291826268561, 'early_stopping_patience': 14}. Best is trial 0 with value: 101.92406972249348.


2025-10-19 05:47:17,745 - __main__ - INFO - Epoch [10/100] - Train Loss: 421.359112, Val Loss: 166.842781
2025-10-19 05:47:18,324 - __main__ - INFO - Epoch [20/100] - Train Loss: 332.727364, Val Loss: 142.649750
2025-10-19 05:47:18,903 - __main__ - INFO - Epoch [30/100] - Train Loss: 283.016503, Val Loss: 111.997537
2025-10-19 05:47:19,491 - __main__ - INFO - Epoch [40/100] - Train Loss: 262.647171, Val Loss: 121.938717
2025-10-19 05:47:20,070 - __main__ - INFO - Epoch [50/100] - Train Loss: 252.856684, Val Loss: 116.901632
2025-10-19 05:47:20,657 - __main__ - INFO - Epoch [60/100] - Train Loss: 226.798064, Val Loss: 109.433159
2025-10-19 05:47:21,283 - __main__ - INFO - Epoch [70/100] - Train Loss: 214.072314, Val Loss: 117.239220
2025-10-19 05:47:21,849 - __main__ - INFO - Early stopping at epoch 79
2025-10-19 05:47:21,851 - __main__ - INFO - Neural Network training completed!
2025-10-19 05:47:21,858 - __main__ - INFO - Training Neural Network on cuda...


...

2025-10-19 06:22:10,542 - __main__ - INFO -   Best CV Score (MSE): 61.754598
2025-10-19 06:22:10,543 - __main__ - INFO -   Best hyperparameters:
2025-10-19 06:22:10,544 - __main__ - INFO -     bootstrap: True
2025-10-19 06:22:10,545 - __main__ - INFO -     ccp_alpha: 0.04849571989073016
2025-10-19 06:22:10,545 - __main__ - INFO -     max_depth: None
2025-10-19 06:22:10,546 - __main__ - INFO -     max_features: 0.5
2025-10-19 06:22:10,547 - __main__ - INFO -     max_leaf_nodes: None
2025-10-19 06:22:10,547 - __main__ - INFO -     min_impurity_decrease: 0.046869315979497034
2025-10-19 06:22:10,548 - __main__ - INFO -     min_samples_leaf: 3
2025-10-19 06:22:10,549 - __main__ - INFO -     min_samples_split: 2
2025-10-19 06:22:10,549 - __main__ - INFO -     min_weight_fraction_leaf: 0.0013671964826997285
2025-10-19 06:22:10,550 - __main__ - INFO -     n_estimators: 360
2025-10-19 06:22:10,550 - __main__ - INFO -     random_state: 42
2025-10-19 06:22:10,551 - __main__ - INFO -     warm_star

Fitting 5 folds for each of 500 candidates, totalling 2500 fits


2025-10-19 06:22:48,005 - __main__ - INFO -   Best CV Score (MSE): 62.638397
2025-10-19 06:22:48,006 - __main__ - INFO -   Best hyperparameters:
2025-10-19 06:22:48,006 - __main__ - INFO -     booster: gbtree
2025-10-19 06:22:48,008 - __main__ - INFO -     colsample_bylevel: 0.764468567013606
2025-10-19 06:22:48,008 - __main__ - INFO -     colsample_bynode: 0.969533849256448
2025-10-19 06:22:48,009 - __main__ - INFO -     colsample_bytree: 0.8993916178868326
2025-10-19 06:22:48,009 - __main__ - INFO -     gamma: 0.49896705526666874
2025-10-19 06:22:48,009 - __main__ - INFO -     grow_policy: lossguide
2025-10-19 06:22:48,010 - __main__ - INFO -     learning_rate: 0.02580515079316858
2025-10-19 06:22:48,010 - __main__ - INFO -     max_bin: 445
2025-10-19 06:22:48,011 - __main__ - INFO -     max_depth: 7
2025-10-19 06:22:48,011 - __main__ - INFO -     max_leaves: 43
2025-10-19 06:22:48,011 - __main__ - INFO -     min_child_weight: 9
2025-10-19 06:22:48,013 - __main__ - INFO -     n_estim

PATH OPTIMIZATION SUMMARY
Direct Path:
  Average RSSI: -101.25 dBm
  Average SNR:  0.49 dB
  Average PDR:  0.5960 (59.60%)
Optimal Path:
  Average RSSI: -100.60 dBm
  Average SNR:  4.15 dB
  Average PDR:  0.7979 (79.79%)
  Minimum PDR:  0.5797 (57.97%)
  Path length:  27 beacons
  Avg Elevation: 31.3 m (from SRTM)
  Avg Terrain : 0.365 (from ESA WorldCover)
  Avg Terrain Penalty: 0.365 (from ESA WorldCover)
Improvements:
  RSSI: +0.66 dBm (+0.65%)
  SNR:  +3.67 dB (+754.02%)
  PDR:  +20.18%


2025-10-19 06:23:47,803 - __main__ - INFO -   Feature importance saved: output\xgboost_feature_importance.png
2025-10-19 06:23:47,804 - __main__ - INFO - Exporting results to CSV...
2025-10-19 06:23:47,808 - __main__ - INFO -   Optimal path saved: output\optimal_path.csv
2025-10-19 06:23:47,815 - __main__ - INFO -   Grid points saved: output\grid_points.csv
2025-10-19 06:23:47,819 - __main__ - INFO -   Direct path saved: output\direct_path.csv
2025-10-19 06:23:47,821 - __main__ - INFO -   Model comparison saved: output\model_comparison.csv
2025-10-19 06:23:47,823 - __main__ - INFO -   Feature importance saved: output\feature_importance.csv
2025-10-19 06:23:47,824 - __main__ - INFO - All CSV exports completed!
2025-10-19 06:23:47,825 - __main__ - INFO - Plotting model comparison...
2025-10-19 06:23:48,096 - __main__ - INFO -   Model comparison saved: output\model_comparison.png
2025-10-19 06:23:48,097 - __main__ - INFO - Plotting path comparison...
2025-10-19 06:23:48,944 - __main__ - I